# E14-A · optimized engine on 4xL4: measurements + a Duck run (SGLang 0.5.21 + RedHat INT4 + DFlash2)

**Base (unchanged):** the E12 rev 6g G4B notebook built in memory by build_e12_duck.py (Duck rev 6g solver, W1 engine).

**E14 changes (Notes/07, 11, 12, 13):** ranged parallel model copy to RAM/local disk (verified, atomic, fail open), a stall-based launch watchdog (log + read bytes + page faults; 75-min give-up kept), JIT cache dir + a cache tar without nv/, the raw-reasoning fix in Duck's history (E12_RAW_REASONING_HISTORY=1), a phase record and heartbeat.

**Commit (profile E14A):** NCCL bench while the copy runs; W1 baseline windows (8k / 40k at 12 really running), profiler, latch test, E14-P token replay, greedy parity (A/A and 12 at once); +#44401 backport; +nodelay (--min-free-slots-delay 1); +CUDA graphs 1..13 - each with a correctness gate, adopted only on every recorded condition; a 25-minute Duck run (reserved) on a fresh launch of the kept arm (G4B tasks); then a matched reference, int8 Mamba checkpoints with an agent-like replay, target-only, DFlash width 6 and 4, the NCCL env arm if the bench found one (each gated). Hard stop at 198 min.

**Competition rerun:** every hidden task with the rev 6g Duck solver + the E14 load path + raw reasoning; engine = W1 + DFlash width 6 + CUDA graphs 1..13 (adopted in R2, Notes/30).

**Commit (profile E14S, the default):** the 25-min Duck run on the G4B tasks with the same engine arm, no measurements. Profile E14A = the R2 measurements.

**Inputs:** the E14 modules and the Duck package come from the private dataset `foysalemonshanto/q27-e14a-payload` (files named by their sha256 and verified before use). It must be attached, for commits and for the submission. Reason: Kaggle commit runs of the 745 kB embedded notebook never started (Notes/21).


In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — configuration
PROFILE = "RERUN_ALL" if RERUN else os.environ.get("Q27X_PROFILE", "E14S").strip().upper()
PROFILE_TABLE = {
    # commit run: 12 public evaluation tasks, scored; also measures throughput for the 12-hour projection
    "EVAL12":    {"source": "eval", "n_tasks": 12, "time_budget_sec": 3.0 * 3600, "admit_frac": 0.80, "drain_frac": 0.90},
    # S1 quick commit: 4 eval tasks that closed fastest in R5 iteration 1 -> a valid version in about 1 h, then submit
    "S1COMMIT4": {"source": "eval", "task_list": ["1818057f", "2ba387bc", "bf45cf4b", "58490d8a"],
                  "time_budget_sec": 90 * 60, "admit_frac": 0.80, "drain_frac": 0.90},
    # R5 EVAL120: full evaluation run (120 tasks / 172 inputs) on the hidden-run clock (RERUN_ALL margins)
    "EVAL120":   {"source": "eval", "n_tasks": 120, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E9/E10: PROXY48 (EDA LB-mirror) + the 8 EVAL12 tasks not in it = 56 tasks / 72 test inputs
    "EVAL14":    {"source": "eval", "task_list": ["135a2760", "136b0064", "20a9e565", "21897d95", "247ef758", "2b83f449", "3dc255db", "4c3d4a41", "4c416de3", "4c7dc4dd", "62593bfd", "64efde09", "65b59efc", "6e4f6532"],
                  "time_budget_sec": 2.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 step 3.5: GPU smoke, 1 fast task (2ba387bc: 5,355 thinking tokens in E10 ORDER2), 2 branches, 15 min solving
    "SMOKE1":    {"source": "eval", "task_list": ["2ba387bc"], "time_budget_sec": 50 * 60, "admit_frac": 0.90,
                  "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60,
                  "admit_margin_sec": 2 * 60, "solve_window_sec": 15 * 60},
    # E12 step 4: the E11-05 EVAL30 tasks (same seed sample), 4 h
    "EVAL30":    {"source": "eval", "n_tasks": 30, "time_budget_sec": 4.0 * 3600,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E12 rev 4 (user 2026-10-20): EVAL6 = 6 tasks that S49/R5/DIRECT solved and E12 EVAL30 did not (accuracy test)
    "EVAL6":     {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd"],
                  "time_budget_sec": 125 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 90 * 60},   # audits: the 100k cap is soft; 12 x ~120k = 1.44 M at 368 tok/s = 65 min + prefill
    # E12 rev 5 (user 2026-10-21): the same 6 tasks with every validated fix (A/B against rev 4 EVAL6: 3/6, 4/8)
    "EVAL6R5":   {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd"],
                  "time_budget_sec": 125 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 90 * 60},
    # E12 rev 5 (user 2026-10-21): EVAL6R5 + 6 informative EVAL30 tasks: 4 Duck rev 1 solved (247ef758 Duck-only on
    # query 1 and first-exact-wrong; bf45cf4b 9.5k; f931b4a8 38.8k; 65b59efc 67k) + 2 mid ones it missed (9385bd28 and
    # cbebaa4b, ledger solve rate 0.25). 24 branches / 6 live; 150 min solving, 185 min budget
    "EVAL12R5":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 185 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 150 * 60},
    # E12 rev 5: the E12 EVAL30 tasks (same seed sample) with every fix (A/B against rev 1 EVAL30 7/48 and Poetiq R5 16/48).
    # Capacity: rev 4 EVAL6 measured about 10.4 min per task at 6 live -> 30 tasks about 5.2 h of solving
    "EVAL30R5":  {"source": "eval", "n_tasks": 30, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94,
                  "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 330 * 60},
    # E12 rev 6 (user 2026-10-21): the EVAL12R5 tasks with rev 6 (A/B: rev 5 xhigh EVAL6R5 5/8, low EVAL12R5 10/19)
    "EVAL12R6":  {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 185 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 150 * 60},
    "EVAL30R6":  {"source": "eval", "n_tasks": 30, "time_budget_sec": 6.0 * 3600, "admit_frac": 0.90, "drain_frac": 0.94,
                  "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 330 * 60},
    # gate 4b (user 2026-10-21): the EVAL12R6 tasks at the hidden run's compute share. CELL 6 sets the solve window to
    # n_tasks x (hidden solve seconds / 240), the hidden solve end taken from RERUN_ALL and the start measured in this run
    "EVAL12R6G4B": {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                    "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                    "time_budget_sec": 80 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                    "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                    "hidden_equiv": True},
    # R6-X1 (user 2026-10-21): target-only vs DFlash2 parity on 4 tasks (no solving): DFlash2 twice (control), then the
    # same server without --speculative-*, the same requests at T 0, one at a time, thinking on and off
    "PARITYR6":  {"source": "eval", "task_list": ["136b0064", "bf45cf4b", "247ef758", "b5ca7ac4"],
                  "time_budget_sec": 75 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "parity": True},
    # E14-A (user 2026-10-23: one notebook; plan Notes/13): engine measurements (A1), a 25-minute Duck run with the
    # adopted levers on the G4B tasks (raw reasoning on), more engine arms (A2). Every phase has its own deadline;
    # whatever does not fit before the hard stop is skipped and reported, never squeezed.
    "E14A":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 200 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 25 * 60, "e14": True},
    # E14S (Notes/30, 31; user 2026-10-24: a short commit): a valid version + a GPU smoke of the submission arm (DFlash
    # width 6 + CUDA graphs 1..13) at 12 streams, no measurements. 36 min, not 30: R2 was READY at ~12.5 min (install
    # 3 + staging 2 + first launch 502 s) and the SGLang give-up is admit - 5 min (30 min -> 13 min, 36 min -> 19 min)
    "E14S":      {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd",
                                                  "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"],
                  "time_budget_sec": 36 * 60, "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60,
                  "solve_window_sec": 8 * 60},
    # E14S EVAL120 gate 4b (user 2026-10-24): every evaluation task (120 / 172 inputs) with the hidden run's compute per
    # task: CELL 6 sets the window to 120 x (hidden solve seconds / 240) = about 5.8 h; the coverage of the hidden run
    # (stage-1 starts, branch 1 starts, pass1 / pass2) on eval difficulty. 6.5 h budget = start + window + margins
    "E14SEVAL120G4B": {"source": "eval", "n_tasks": 120, "time_budget_sec": 6.5 * 3600, "admit_frac": 0.90,
                       "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                       "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "hidden_equiv": True},
    "PROXY56":   {"source": "eval", "task_list": ["0934a4d8", "20a9e565", "4c7dc4dd", "89565ca0", "136b0064", "291dc1e1", "16b78196", "271d71e2", "28a6681f", "2b83f449", "2c181942", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4c416de3", "64efde09", "6e453dd6", "7491f3cf", "7b0280bc", "7b80bb43", "7c66cb00", "1818057f", "332f06d7", "53fb4810", "7666fa5d", "da515329", "8f3a5a89", "db695cfb", "97d7923e", "de809cff", "c7f57c3e", "cb2d8a2c", "58490d8a", "dbff022c", "5961cc34", "80a900e0", "269e22fb", "8b9c3697", "e376de54", "b99e7126", "c4d067a0", "135a2760", "4a21e3da", "71e489b6", "800d221b", "dd6b8c4b", "36a08778", "21897d95", "247ef758", "4c3d4a41", "6e4f6532", "88bcf3b4", "898e7135", "8b7bacbf", "b6f77b65"], "time_budget_sec": 6.0 * 3600,
                  "admit_frac": 0.80, "drain_frac": 0.90},
    # hidden test: every task; Kaggle's limit is 12 h including install + model load
    # v1.4.3: stop 10 min before 12 h (user); hard stop 6 min before the budget end (outputs take seconds after it)
    "RERUN_ALL": {"source": "test", "n_tasks": None, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  # v1.4.4: admit until the end (NVARC style); drain 8 min before the hard stop, admit 2 min before drain
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
}
if PROFILE not in PROFILE_TABLE:
    raise ValueError(f"unknown profile {PROFILE}; choose {list(PROFILE_TABLE)}")
P = PROFILE_TABLE[PROFILE]
SEED = 20261005
TASK_SAMPLE_SEED = 20261005

MODEL_PATH = Path(os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
WHEELHOUSE = Path(os.environ.get("Q27X_WHEELHOUSE", "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"))

# ---- generation budget (Kaggle discussion 745787: Qwen3.8 needs xhigh; median ~64k tokens; forced commit ~49k)
REASONING_EFFORT = "xhigh"
THINK_BUDGET = int(os.environ.get("Q27X_THINK_BUDGET", 60000))   # E11 rev 3: 60,000 = S1-60K (LB 34.58); user floor 60k   # S49: LB test of 49,152 (S1: 60,000; R5/E9: 77,824; floor 49,152)      # v1.2: NguyenThanhNhan forced the commit after ~49k thinking tokens
E11_THINK_FLOOR = 60000      # E11 rev 3 (user 2026-10-18): never below 60k (budget and admission guard)
assert THINK_BUDGET >= E11_THINK_FLOOR, (THINK_BUDGET, E11_THINK_FLOOR)
ANSWER_MAX_TOKENS = int(os.environ.get("Q27X_ANSWER_MAX", 8192))   # v1.4: grid-only answer; 49,152 + 8,192 = 57,344 = russcore's max_tokens
DRAIN_ANSWER_MAX_TOKENS = 8192                                      # v1.1 (F11): a 30x30 grid alone is ~2.7k tokens
STOP_THINKING_TEXT = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
MAX_MODEL_LEN = int(os.environ.get("Q27X_MAX_MODEL_LEN", 106496))   # E9/E10: prompt (<= ~20k) + 77,824 + 8,192 fits; was 81,920   # v1.2: native context is 262k; fp8 KV makes room
CONTEXT_SAFETY = 256
MIN_THINK_BUDGET = 4096                  # v1.2: a long prompt gets a smaller thinking budget instead of being skipped
PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
def phase1_cap(prompt_len):
    return max(0, min(THINK_BUDGET, MAX_MODEL_LEN - prompt_len - ANSWER_MAX_TOKENS - CONTEXT_SAFETY))
assert phase1_cap(MAX_MODEL_LEN) == 0 and phase1_cap(100) == min(THINK_BUDGET, MAX_MODEL_LEN - 100 - ANSWER_MAX_TOKENS - CONTEXT_SAFETY)
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (106496, 77824, 8192):   # E9/E10
    assert phase1_cap(12800) == 77824 and phase1_cap(20000) == 77824 and phase1_cap(30000) == 68048
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (81920, 49152, 8192):
    assert phase1_cap(24320) == 49152 and phase1_cap(30000) == 43472  # every real prompt (≤ ~9k tokens) keeps the full budget
SAMPLING = {"temperature": float(os.environ.get("Q27X_TEMPERATURE", 0.6)), "top_p": 0.95, "top_k": 20}   # v1.4: russcore LB-10.14 sampling

# ---- engine (PIV/V3.1 C1 settings; context, concurrency and MTP changed)
MAX_NUM_SEQS = int(os.environ.get("Q27X_MAX_NUM_SEQS", 16))   # upper bound; actual slots = KV tokens // EXPECTED_LIVE_TOKENS_PER_SEQ
MAX_NUM_BATCHED_TOKENS = 4096   # v1.3.5: = discussion max-prefill-tokens 4096; 32768 ran out of memory in run 2 (MTP drafter)
GPU_MEMORY_UTILIZATION = float(os.environ.get("Q27X_GPU_MEM", 0.90))   # v1.4.1: kept at 0.90 so prefix caching is the only change
ENABLE_PREFIX_CACHING = os.environ.get("Q27X_PREFIX_CACHE", "1") == "1"   # v1.4.1: first engine attempt (mamba 'align')
SPECULATIVE_CONFIG = {"method": "mtp", "num_speculative_tokens": int(os.environ.get("Q27X_SPEC_TOKENS", 3))}   # v1.2: vLLM Qwen3.8 recipe uses 3
KV_CACHE_DTYPE = os.environ.get("Q27X_KV_DTYPE", "fp8_e4m3")   # v1.3.4: back to the discussion recipe; run 1's crash was
# FlashInfer's JIT link step (`ld: cannot find -lcuda`), fixed in cell 5 (libcuda shim + link probe); fp8 KV itself loaded
# fine (312,000 KV tokens). Set Q27X_KV_DTYPE=auto for bf16 KV.
if os.environ.get("Q27X_NO_SPEC") == "1":
    SPECULATIVE_CONFIG = None
EXPECTED_LIVE_TOKENS_PER_SEQ = 43000      # a running task holds on average about half of its final ~60k length
# v1.3.5 correction: cell 5 divides the whole KV pool (run 2: 781 blocks × 1,600 = 1,249,600 tokens) by this. The pool is
# shared by 1 attention + 3 GDN groups; vLLM's "312,000 tokens" log line is pool // 4. A 30k-token task needs 19 attention
# + 12 GDN-state blocks (3 groups × (1 + 3 MTP)), so 16 tasks need ~496 of 781 blocks -> 16 slots (preemptions reported)

# ---- verification (fresh process per candidate: train first, then test)
VERIFY_TIMEOUT_SEC = 20.0
VERIFY_WORKERS = 4

# ---- time
TOTAL_BUDGET_SEC = float(os.environ.get("Q27X_TIME_BUDGET", P["time_budget_sec"]))
# v1.3: deadlines derived in a guaranteed order (v1.2 had DRAIN after HARD_STOP for EVAL12, so the drain never ran)
VERIFY_DEADLINE = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("verify_margin_sec", 8 * 60)   # reporting only since v1.4
HARD_STOP = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("hard_margin_sec", 25 * 60)        # abort everything, write outputs
DRAIN_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["drain_frac"], HARD_STOP - 8 * 60)    # force-close thinking
ADMIT_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["admit_frac"], DRAIN_DEADLINE - 7 * 60)  # no new task after this
if "drain_margin_sec" in P:   # v1.4.4 (RERUN): fixed offsets from the hard stop, so the slots stay full until the drain
    DRAIN_DEADLINE = HARD_STOP - P["drain_margin_sec"]
    ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP < VERIFY_DEADLINE < NOTEBOOK_T0 + TOTAL_BUDGET_SEC

# v1.3.5 (run 2 OOM): CUDA-graph memory is taken out of the KV budget (vLLM 0.19 envs.py 1654, v1/worker/gpu_worker.py
# 393-420); run 2 captured 0.61 GiB of graphs beyond it. Set here, before cell 5 spawns the vLLM workers (they inherit it).
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
# ---- E11-05: engine = SGLang 0.5.21 + RedHat INT4 (W4A16 g128) + DFlash2 drafter = the E11-04 W1 launch (2026-10-18).
# MODEL_PATH (FP8) stays: its tokenizer + chat template render every prompt exactly as in S49, and it is the vLLM fallback.
ENGINE_BACKEND = os.environ.get("E11_ENGINE", "sglang").strip().lower()       # sglang (vLLM FP8 fallback) | vllm (= S49)
E11_ALLOW_VLLM_FALLBACK = os.environ.get("E11_VLLM_FALLBACK", "1") == "1"
SGL_MODEL_PATH = Path(os.environ.get("E11_INT4", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-int4-redhat/pytorch/w4a16-g128/1"))
SGL_DRAFT_PATH = Path(os.environ.get("E11_DFLASH", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-dflash2/pytorch/bf16/1"))
SGL_WHEELHOUSE = Path(os.environ.get("E11_SGL_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/sglang-0-5-21-cu13-wheelhouse"))


def _e11_discover(path, marker, pred):
    """The given path if it holds `marker`, else the first /kaggle/input folder whose `marker` json satisfies pred."""
    if (path / marker).is_file():
        return path
    root = Path("/kaggle/input")
    for p in (sorted(root.rglob(marker)) if root.exists() else []):
        try:
            if pred(p.parent, _load_json(p)):
                return p.parent
        except Exception:
            continue
    return path


SGL_MODEL_PATH = _e11_discover(SGL_MODEL_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and (c.get("quantization_config") or {}).get("quant_method") == "compressed-tensors")
SGL_DRAFT_PATH = _e11_discover(SGL_DRAFT_PATH, "config.json", lambda d, c: isinstance(c, dict)
                               and c.get("architectures") == ["DFlash2DraftModel"])
SGL_WHEELHOUSE = _e11_discover(SGL_WHEELHOUSE, "BUILD_INFO.json", lambda d, c: isinstance(c, dict)
                               and c.get("sglang") == "0.5.21" and (d / "requirements.lock").is_file())
SGL_CONTEXT_LEN = 155648            # = the E11-04 W1 launch; every request stays inside MAX_MODEL_LEN (phase1_cap)
SGL_MAX_RUNNING = 12                # E11-04 W1: 12 really running (mamba cache 56 / 4 state slots = 14 >= 12)
SGL_SERVER_MAX_RUNNING = 13         # rev 3c: server admission limit = client 12 + 1. SGLang 0.5.21 counts the row of a request in
                                    # chunked prefill twice (scheduler.py:3971-3976) and latches batch_is_full (3846-3848) until a
                                    # running request ends -> EVAL30 ran 11 + 1 queued (2,358 of 2,509 decode lines). Upstream fix
                                    # PR #44401 (after 0.5.21); +1 is the documented workaround (PR #30295). CUDA graphs stay at 12.
SGL_LAUNCH_TIMEOUT_SEC = 30 * 60    # first launch includes JIT + CUDA graphs: 567 s in E11-04
SGL_RELAUNCH_TIMEOUT_SEC = 20 * 60  # a diagnosed relaunch (E11-04: about 75 s once the JIT cache exists)
SGL_GIVE_UP_AT = min(NOTEBOOK_T0 + 75 * 60, ADMIT_DEADLINE - 5 * 60)   # no SGLang launch after this -> vLLM fallback
SGL_STREAM_IDLE_SEC = 1800          # a socket read silent for longer closes that request as 'error'
# rev 2 (GPT audit 2026-10-18, verified): capacity gate from the server log, fail-closed on missing evidence
SGL_MIN_RUNNING = 8                 # below this the SGLang engine is refused (-> S49 vLLM)
SGL_MIN_KV = 600_000                # KV tokens; E11-04 W1 had 881,336
SGL_KV_PER_REQUEST = 70_000         # concurrency <= KV // this (881,336 -> 12); a long Poetiq unit is prompt + 49k + 8k
SGL_MIN_FREE_GB = 1.0               # GPU headroom after start-up (E11-04 W1: 2.22 GB)
# rev 2: drain earlier. Aborted long prefixes were not cached in E11-04 (30k prefix: 0 cached, 24.5 s), so 12 phase-2
# continuations re-prefill up to ~0.7M tokens; 12 min instead of 8 min before the hard stop (SGLang only)
SGL_DRAIN_MARGIN_SEC = 8 * 60      # E11-06: back to 8 min (EVAL30 drain <= 83 s)
# rev 2: KV gate active on SGLang (S49's floor 12 = CONCURRENCY 12 made it inert): from 10 running, admit only below 85 %
SGL_KV_GATE_MIN_ACTIVE = 10
SGL_KV_ADMIT_MAX = 0.85

CONFIG = {k: globals()[k] for k in ["PROFILE", "RERUN", "SEED", "REASONING_EFFORT", "THINK_BUDGET", "ANSWER_MAX_TOKENS",
          "DRAIN_ANSWER_MAX_TOKENS", "MAX_MODEL_LEN", "PROMPT_TOKEN_LIMIT", "SAMPLING", "MAX_NUM_SEQS",
          "MAX_NUM_BATCHED_TOKENS", "GPU_MEMORY_UTILIZATION", "ENABLE_PREFIX_CACHING", "SPECULATIVE_CONFIG", "KV_CACHE_DTYPE", "VERIFY_TIMEOUT_SEC",
          "TOTAL_BUDGET_SEC", "MIN_THINK_BUDGET"]}
CONFIG["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"]
CONFIG["profile_row"] = P
CONFIG["e11"] = {k: (str(globals()[k]) if isinstance(globals()[k], Path) else globals()[k]) for k in (
    "ENGINE_BACKEND", "E11_ALLOW_VLLM_FALLBACK", "SGL_MODEL_PATH", "SGL_DRAFT_PATH", "SGL_WHEELHOUSE", "SGL_CONTEXT_LEN",
    "SGL_MAX_RUNNING", "SGL_SERVER_MAX_RUNNING", "SGL_LAUNCH_TIMEOUT_SEC", "SGL_RELAUNCH_TIMEOUT_SEC", "SGL_STREAM_IDLE_SEC", "SGL_MIN_RUNNING",
    "SGL_MIN_KV", "SGL_KV_PER_REQUEST", "SGL_MIN_FREE_GB", "SGL_DRAIN_MARGIN_SEC", "SGL_KV_GATE_MIN_ACTIVE", "SGL_KV_ADMIT_MAX")}
CONFIG["e11"]["give_up_after_sec"] = round(SGL_GIVE_UP_AT - NOTEBOOK_T0)
print("E11 engine:", json.dumps(CONFIG["e11"]))
print(json.dumps(CONFIG, indent=1, default=str))

os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"


# ---- E10 Poetiq donor configurations; source: poetiq-ai/poetiq-arc-agi-solver/config.py
# Original 86K context, GPU recipe, deadlines and original puzzle order above are unchanged.
POETIQ_NUM_EXPERTS = int(os.environ.get('E10_POETIQ_EXPERTS','1'))
POETIQ_MAX_ITERATIONS = int(os.environ.get('E10_POETIQ_MAX_ITERATIONS','10'))
POETIQ_MAX_SOLUTIONS = int(os.environ.get('E10_POETIQ_MAX_SOLUTIONS','2'))
POETIQ_SOLVER_TEMPERATURE = float(os.environ.get('E10_POETIQ_TEMPERATURE','1.0'))
POETIQ_SOLVER_VARIANTS = tuple(int(v) for v in os.environ.get('E10_POETIQ_PROMPT_VARIANTS','1').split(','))
POETIQ_SELECTION_PROB = float(os.environ.get('E10_POETIQ_SELECTION_PROBABILITY','1.0'))
POETIQ_SEED = int(os.environ.get('E10_POETIQ_SEED','0'))
POETIQ_SHUFFLE_EXAMPLES = os.environ.get('E10_POETIQ_SHUFFLE','0')=='1'
POETIQ_IMPROVING_ORDER = os.environ.get('E10_POETIQ_IMPROVING_ORDER','1')=='1'
POETIQ_USE_NEW_VOTING = os.environ.get('E10_POETIQ_NEW_VOTING','1')=='1'
POETIQ_COUNT_FAILED_MATCHES = os.environ.get('E10_POETIQ_COUNT_FAILED_MATCHES','1')=='1'
POETIQ_ITERS_TIEBREAK = os.environ.get('E10_POETIQ_ITERS_TIEBREAK','0')=='1'
POETIQ_LOW_TO_HIGH_ITERS = os.environ.get('E10_POETIQ_LOW_TO_HIGH_ITERS','0')=='1'
POETIQ_SANDBOX_TIMEOUT = float(os.environ.get('E10_POETIQ_SANDBOX_TIMEOUT','5'))
POETIQ_SANDBOX_WORKERS = int(os.environ.get('E10_POETIQ_SANDBOX_WORKERS','4'))
# S1: KV admission gate (no new request above this vLLM KV usage unless fewer than MIN_ACTIVE are running)
E10_KV_ADMIT_MAX = float(os.environ.get('E10_KV_ADMIT_MAX','0.95'))
E10_KV_GATE_MIN_ACTIVE = int(os.environ.get('E10_KV_GATE_MIN_ACTIVE','12'))
assert POETIQ_NUM_EXPERTS >= 1 and POETIQ_MAX_ITERATIONS >= 1
assert POETIQ_SOLVER_VARIANTS and all(v in (1,2,3) for v in POETIQ_SOLVER_VARIANTS)
assert POETIQ_MAX_SOLUTIONS >= 1 and 0 <= POETIQ_SELECTION_PROB <= 1
CONFIG['poetiq']={k:globals()[k] for k in (
    'POETIQ_NUM_EXPERTS','POETIQ_MAX_ITERATIONS','POETIQ_MAX_SOLUTIONS',
    'POETIQ_SOLVER_TEMPERATURE','POETIQ_SOLVER_VARIANTS','POETIQ_SELECTION_PROB',
    'POETIQ_SANDBOX_TIMEOUT','POETIQ_SANDBOX_WORKERS','POETIQ_USE_NEW_VOTING')}


# E16 (2026-10-24): one early inspection-only nudge; raw assistant content whitespace kept only when the
# normalized text is identical (tool-call markup never modified). E12_E16_EARLY_NUDGE=0 / E12_E16_RAW_CONTENT=0 disable.
# Threshold 12,000 (user 2026-10-24; R2 replay, agent C: 4k lands before the natural transform in all 4 solved branches,
# 12k spares b5ca7ac4 and keeps bf45cf4b_b0 and the never-transform branches)
E16_FLAGS = {"E12_E16_EARLY_NUDGE": 1, "E12_E16_INSPECTION_CALLS": 2, "E12_E16_NUDGE_TOKENS": 12000,
             "E12_E16_RAW_CONTENT": 1}
for _e16_key, _e16_val in E16_FLAGS.items():
    os.environ.setdefault(_e16_key, str(_e16_val))
# ---- E12: Duck REPL agent (Experiment 12, user option A 2026-10-19). Duck's own settings are in arc2_duck.DUCK_ENV
# (= dfranzen milestone-2 cell 4): 12,288 output per request, yield at 2,048 generated tokens, preserved thinking,
# 128 Ki context with drain to 58 Ki. Approved exception to the 60k-per-request floor (thinking accumulates in context).
E12_BRANCHES = int(os.environ.get("E12_BRANCHES", 2))                 # independent branches per task -> attempt_1/2
_E12_REV5 = True                                # rev 5 build: every profile (incl. the hidden rerun) runs the fixes
_E12_REV6 = True                                # rev 6 build (user 2026-10-21): code-first view + coverage scheduler
_E12_REV4 = PROFILE == "EVAL6" or _E12_REV5                            # rev 4 defaults (pre-run capacity check, 2026-10-20)
E12_STREAMS = int(os.environ.get("E12_STREAMS", 12 if _E12_REV4 else 0))   # Duck ARC3_MAX_ACTIVE_STREAMS; 0 = engine concurrency
E12_MAX_GEN_PER_BRANCH = int(os.environ.get("E12_MAX_GEN", 100000 if _E12_REV4 else 0)) or None   # Duck max_generated_tokens_per_game
# rev 4: at most N branches alive (resident contexts <= KV 867k / 128k context = 6; EVAL30: 30 live -> 8.4 M re-prefill),
# branches of a task start together; and up to 60,000 output per request (EVAL30: 29/30 first requests cut at 12,288)
E12_LIVE_BRANCHES = int(os.environ.get("E12_LIVE", 12 if _E12_REV4 else 0)) or None   # 6 x 131k = 786k <= KV 867k; S12 (user): 12 x rev 5 context p50 58k = 697k
# rev 5 B5 (evidence, 2026-10-21): no productive EVAL6 reply exceeded 44,779 tokens; the 7 replies that hit 60,000 made no
# tool call; at 49,152 the input budget is 81,408 (4 of the 6 real-overflow trims would have fitted). 40k would have cut one.
E12_DUCK_MAX_OUTPUT = int(os.environ.get("E12_DUCK_MAX_OUTPUT", (12288 if _E12_REV6 else 49152 if _E12_REV5 else 60000)
                                         if _E12_REV4 else 0)) or None   # rev 6: the first-request cap; later caps are per request
# rev 5 B9 (user 2026-10-21): Qwen3.8 model-card thinking temperature 1.0 (Duck milestone 0.7)
E12_DUCK_TEMPERATURE = float(os.environ.get("E12_DUCK_TEMPERATURE", 1.0 if _E12_REV5 else 0)) or None
E12_REASONING_EFFORT = os.environ.get("E12_REASONING_EFFORT", "").strip().lower()   # "" = model default (xhigh)
if E12_REASONING_EFFORT:   # user 2026-10-21: low-effort variant of rev 5 (Qwen3.8 template: xhigh / medium / low)
    assert E12_REASONING_EFFORT in ("xhigh", "medium", "low"), E12_REASONING_EFFORT
    os.environ["E12_REASONING_EFFORT"] = E12_REASONING_EFFORT
if _E12_REV5:   # B1-B4, B6-B8 in arc2_duck (E12_ARC2_REV5); B3 counts reasoning with the served model's tokenizer
    os.environ["E12_ARC2_REV5"] = "1"
    os.environ["E12_TOKENIZER_JSON"] = str(SGL_MODEL_PATH / "tokenizer.json")
# rev 6 (user 2026-10-21; fix list "Rev 6 gap check", both plan reviews): facts view + trusted helpers + auto-check +
# state-aware resumes (arc2_duck E12_ARC2_REV6); request caps 12,288 -> 24,576 after the first successful Python call,
# 32,768 after two no-call cuts (thinking kept in history, branch allowance >= 60k, soft cap 100k); first exposure: branch 0
# of every task in ORDER2 order, paused at about 28k generated tokens, then stage 2 by evidence (resume / branch 1 with the
# grid view = the representation hedge); 10 workers, context window sized to the KV pool in CELL 6; per-request deadline.
E12_R6 = {"E12_R6_CAP_FIRST": 12288, "E12_R6_CAP_PROGRESS": 24576, "E12_R6_CAP_RESCUE": 32768,
          "E12_R6_FIRST_EXPOSURE": 28000, "E12_R6_B1_VIEW": "indexed", "E12_R6_MIN_ADMIT_SEC": 240,
          "E12_CHECK_VERIFY_CAP": 24, "E12_AUTOCHECK_CAP": 24, "E12_AUTOCHECK_PER_CALL": 3, "E12_DUCK_DRAIN_TOKENS": 16384,
          # rev 6c (user 2026-10-21: 12 streams): the per-stream window = 1.0 x KV / 12 (= 73.4k at 881k). 0.9 would give
          # 66.1k (history 41.5k at the 24,576 cap); D66 ran 12 slots at context 155,648 with KV peak 90 % and 0 preemptions,
          # rev 6e G4-1 (user 2026-10-21): workers stay 12; no new task or branch while KV usage >= 0.95
          "E12_R6_FACTS_MAX_CHARS": 3600, "E12_R6_SHOW_MAX_CELLS": 900, "E12_R6_KV_FRACTION": 1.0,
          "E12_R6_P90_CUTOFF": 0,
          # rev 6b (gap check vs the problem->fix map): facts <= 800 Qwen tokens (Q-5; 120 eval tasks: full p50 1,237 ->
          # bounded p50 736 / p90 837 / max 1,396), fair-share stage 2 (R6-S3) with a 24,576-token minimum chunk
          "E12_R6_FACTS_MAX_TOKENS": 800, "E12_R6_FAIR_SHARE": 1,
          # rev 6e (user 2026-10-21: "take all", fix list G4-1..G4-18; G4B evidence): KV gate 0.95 polled every 5 s (G4-1);
          # branch-1 first cap 24,576 (G4-2; G4B b1 first calls at 12.4-18.0k); show() up to 900 cells per grid, 1,000 per
          # call (G4-4; 30x30 = 2,033 Qwen tokens of the 3,072 tool cap); provisional nudge (G4-6: 27,000 in rev 6e; 24,000 since rev 6f, H6-1; G4B
          # solved branches defined transform at 18,383 / 26,228); branch-1 indexed view <= 12,000 tokens (G4-11);
          # minimum stage-2 chunk 12,288 (G4-17: every G4B share 3.8-7.5k was below 24,576); cache diagnostic (G4-8)
          "E12_R6_KV_ADMIT": 0.95, "E12_R6_KV_POLL": 5.0, "E12_R6_CAP_FIRST_B1": 24576, "E12_R6_SHOW_CALL_CELLS": 1000,
          "E12_R6_NUDGE_TOKENS": 24000, "E12_R6_B1_GRID_MAX_TOKENS": 12000, "E12_R6_MIN_CHUNK": 12288,
          "E12_R6_CACHE_DIAG": 1}
# rev 6f (user 2026-10-22, the H6 set of the three GPT rev 6e audits): the provisional nudge at 24,000 tokens, sent in the
# next tool result (H6-1: in G4B no branch 0 started a new turn at >= 27k before the 28k pause); the cache diagnostic
# runs AFTER solving (H6-3); every commit window starts after the preflight (H6-6); one isolation retry (H6-17)
if _E12_REV6:
    os.environ["E12_ARC2_REV6"] = "1"
    for _k, _v in E12_R6.items():
        E12_R6[_k] = type(_v)(os.environ.get(_k, _v))
        os.environ[_k] = str(E12_R6[_k])
# audits (verified): non-streaming read timeout 900 s < a 60k reply at ~61 tok/s per stream (978 s) -> 1,800 s.
# The 100k per-branch cap is soft (checked after each request). max_tokens 60,000 is a ceiling, not a minimum thinking.
# The milestone context (131,072, drain 58 Ki) is kept: input budget 70,560, trim target 35,280. Only the last message is protected:
# a tool-less reply before the resume opener could be dropped (EVAL6: 5 resets; rev 5 B3 counts reasoning correctly).
E12_DUCK_REQUEST_TIMEOUT = int(os.environ.get("E12_DUCK_REQUEST_TIMEOUT", 1800 if _E12_REV4 else 0)) or None
for _k, _v in (("E12_DUCK_MAX_OUTPUT", E12_DUCK_MAX_OUTPUT), ("E12_DUCK_REQUEST_TIMEOUT", E12_DUCK_REQUEST_TIMEOUT),
               ("E12_DUCK_TEMPERATURE", E12_DUCK_TEMPERATURE)):
    if _v:
        os.environ[_k] = str(_v)                                         # read by arc2_duck (CELL 3 / CELL 6)
E12_SOLVE_WINDOW_SEC = P.get("solve_window_sec")                      # SMOKE1 only
E12_STOP_BEFORE_HARD_SEC = 180   # branches stop; outputs are written after it. rev 3 (GPT re-audit): a submit at the
                                 # deadline can still verify for <= 60 s, then CELL 6/7 write; 60 s left no margin
E12_MAMBA_TRACK_64 = os.environ.get("E12_MAMBA_TRACK_64", "0") == "1"  # Duck's --mamba-track-interval/--page-size 64 (measure)
# rev 2 (GPT audits, validated 2026-10-19): Duck request snapshots (the exact messages sent; large) for the smoke only,
# and a per-sandbox-process address-space limit (Duck sets CPU / file size / open files only)
E12_REQUEST_LOGS = os.environ.get("E12_REQUEST_LOGS", "1" if PROFILE == "SMOKE1" else "0") == "1"
E12_SANDBOX_MEM_MB = int(os.environ.get("E12_SANDBOX_MEM_MB", 4096))
CONFIG["e12"] = {k: globals()[k] for k in ("E12_BRANCHES", "E12_STREAMS", "E12_MAX_GEN_PER_BRANCH", "E12_SOLVE_WINDOW_SEC",
                                           "E12_LIVE_BRANCHES", "E12_DUCK_MAX_OUTPUT", "E12_DUCK_REQUEST_TIMEOUT",
                                           "E12_DUCK_TEMPERATURE", "_E12_REV5", "E12_REASONING_EFFORT",
                                           "E12_STOP_BEFORE_HARD_SEC", "E12_MAMBA_TRACK_64", "E12_REQUEST_LOGS",
                                           "E12_SANDBOX_MEM_MB", "_E12_REV6")}
if _E12_REV6:
    CONFIG["e12"]["rev6"] = dict(E12_R6)
CONFIG["e12"]["duck_commit"] = "10882e332d883af3e61616df8c8a17d03a328a75"
print("E12:", json.dumps(CONFIG["e12"]))


In [ ]:
# ==================================================================== CELL 1b — E14: first-line facts, phase record, staging
# Experiment 14 (Notes/07, 11-13). Runs before any heavy work: what the machine is (mounts / GCSFUSE, df, RAM, cgroup,
# CPU, a timed 64 MiB read), a phase record written at every change (e14_state.json) and a heartbeat, then the model
# copy to RAM/local disk with ranged parallel readers (verified, published atomically, fail open). The E14-P fix (raw
# reasoning in Duck's history, Notes/12) is switched on for every profile of this notebook.
import io as _e14_io
E14_DIR = Path(os.environ.get("E14_DIR", "/tmp/e14")); E14_DIR.mkdir(parents=True, exist_ok=True)
Q27X_PAYLOAD_SLUG = "foysalemonshanto/q27-e14a-payload"


def q27x_payload(name, sha):
    """One file of the payload dataset (Notes/21: Kaggle commit runs of a 745 kB notebook never started, so the big
    payloads live in a dataset), found by its exact name under Q27X_PAYLOAD_DIR or /kaggle/input (at most 3 directory
    levels deep) and verified by sha256 before use."""
    roots = [Path(x) for x in os.environ.get("Q27X_PAYLOAD_DIR", "").split(os.pathsep) if x] + [Path("/kaggle/input")]
    seen = []
    for r in roots:
        for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name):
            for f in (sorted(r.glob(pat)) if r.is_dir() else []):
                b = f.read_bytes()
                if hashlib.sha256(b).hexdigest() == sha:
                    return b
                seen.append(str(f))
    raise FileNotFoundError(f"payload {name} (sha256 {sha[:12]}) not found or wrong under {[str(r) for r in roots]}; "
                            f"seen {seen}; attach the dataset {Q27X_PAYLOAD_SLUG}")


_E14_MOD_SHA = {"e14_loadpath.py": "7c66c179e5bb6bff7c909d0a96d4b7058c4bc0b2ff368b04367ee9f54e8fc5fc", "e14_measure.py": "ae61c9f9a03419d73dae2ba2d2aa4d5d1f4403780911832148d2dd8e8a3440d0", "e14_sglang_patches.py": "98da95210bb769ca1d7135a4868a01e98f4d2377cd0e15c647aa7c989f3e87d6", "e14_nccl_bench.py": "355adda0d469c5ecac5abb94ff7608e2f256acc6a5dcf745e28b4d3676f03cb2", "e14_trace_summary.py": "8d89ca106e166c97204f1aeff931a50a75e203999c876f248b34e6ee24212b46"}
with zipfile.ZipFile(_e14_io.BytesIO(q27x_payload("q27x_e14_modules_78029e146cd3.bin", "78029e146cd3e1aa8547b9a08ff36c369369af645914203d647c1713ecfa49ae"))) as _z:
    for _n, _h in _E14_MOD_SHA.items():
        (E14_DIR / _n).write_bytes(_z.read(_n))
        assert hashlib.sha256((E14_DIR / _n).read_bytes()).hexdigest() == _h, _n
sys.path.insert(0, str(E14_DIR))
import e14_loadpath as E14L, e14_measure as E14M, e14_sglang_patches as E14P, e14_trace_summary as E14T
E14_OUT = OUT / "e14"; E14_OUT.mkdir(parents=True, exist_ok=True)
E14_RESULTS = {"build": {"variant": "E14S", "arc2_duck_sha": "758bddb5aeeb437d059f70a8de41451b4e00e18adde6a251b66d901ec53a40da", "modules": _E14_MOD_SHA,
                         "payload": {"dataset": Q27X_PAYLOAD_SLUG, "q27x_e14_modules_78029e146cd3.bin": "78029e146cd3e1aa8547b9a08ff36c369369af645914203d647c1713ecfa49ae",
                                     "q27x_e12_duck_6f48b81af59a.bin": "6f48b81af59ad9e1d5d8a94092cc86d017f036d2710f33280838f0b33689c35b"}},
               "phases": {}, "decisions": {}}
E14_DO_MEASURE = bool(P.get("e14")) and not RERUN
E14_STATE = E14L.E14State(WORKING / "e14_state.json")
E14_STATE.enter("PREPARE", deadline=HARD_STOP, profile=PROFILE, rerun=RERUN, measure=E14_DO_MEASURE)


def e14_save():
    try:
        write_json_atomic(E14_OUT / "e14_results.json", E14_RESULTS)
        write_json_atomic(WORKING / "e14_results.json", E14_RESULTS)
    except Exception as _e:
        print("E14 results write failed:", repr(_e)[:200], flush=True)


def _e14_heartbeat(every=60):
    while True:
        try:
            with open(WORKING / "e14_heartbeat.log", "a", encoding="utf-8") as _f:
                _f.write(f"{time.strftime('%H:%M:%S')} +{(time.perf_counter() - NOTEBOOK_T0) / 60:.1f} min "
                         f"phase {E14_STATE.rec.get('current')}\n")
        except Exception:
            pass
        time.sleep(every)


threading.Thread(target=_e14_heartbeat, daemon=True).start()
E14_RESULTS["env"] = E14L.env_report([str(SGL_MODEL_PATH), str(SGL_DRAFT_PATH), str(SGL_WHEELHOUSE), str(MODEL_PATH),
                                      "/kaggle/input"])
try:
    E14_RESULTS["read_probe"] = E14L.read_probe(SGL_MODEL_PATH) if ENGINE_BACKEND == "sglang" else None
except Exception as _e:
    E14_RESULTS["read_probe"] = {"error": repr(_e)[:200]}
print("E14 env:", json.dumps(E14_RESULTS["env"])[:1500], "| probe:", json.dumps(E14_RESULTS["read_probe"]), flush=True)
E14_STAGING = None
if ENGINE_BACKEND == "sglang" and E14L.loadpath_enabled():
    _e14_dest = tuple(x for x in os.environ.get("E14_DEST", "/dev/shm,/tmp").split(",") if x)
    E14_STAGING = E14L.Staging([SGL_MODEL_PATH, SGL_DRAFT_PATH], workers=24, stall_sec=600.0,
                               deadline=NOTEBOOK_T0 + 45 * 60, log=print, state=E14_STATE, dest_candidates=_e14_dest)
    E14_STAGING.start()
    print("E14 staging started ->", _e14_dest, flush=True)
os.environ["E12_RAW_REASONING_HISTORY"] = "1"          # E14-P (Notes/12): Duck keeps the server's raw reasoning
E14_VENV = {}


def e14_after_install(vpy, env):
    """After the SGLang venv exists: the NCCL bench (commit measurement only; the GPUs are idle while the copy runs),
    then the staged model paths (fail open: the original /kaggle/input paths)."""
    global SGL_MODEL_PATH, SGL_DRAFT_PATH
    E14_VENV.update(vpy=str(vpy), env=dict(env))
    if E14_DO_MEASURE and not E11_HARNESS_E14:
        E14_STATE.enter("NCCL_BENCH")
        _t = time.perf_counter()
        _o = E14_OUT / "nccl_bench.json"
        try:
            _r = subprocess.run([str(vpy), str(E14_DIR / "e14_nccl_bench.py"), "--out", str(_o),
                                 "--arms", "baseline,p2p_sys,shm_memcpy", "--timeout-per-arm", "240"],
                                env=env, capture_output=True, text=True, timeout=900)
            E14_RESULTS["nccl"] = json.loads(_o.read_text()) if _o.exists() else {"rc": _r.returncode,
                                                                                  "stderr": (_r.stderr or "")[-2000:]}
        except Exception as _e:
            E14_RESULTS["nccl"] = {"error": repr(_e)[:300]}
        E14_RESULTS["nccl_sec"] = round(time.perf_counter() - _t, 1)
        e14_save()
        print("E14 NCCL bench:", json.dumps((E14_RESULTS.get("nccl") or {}).get("summary"))[:800], flush=True)
    elif E14_DO_MEASURE and E11_HARNESS_E14:
        _o = E14_OUT / "nccl_bench.json"
        subprocess.run([sys.executable, str(E14_DIR / "e14_nccl_bench.py"), "--out", str(_o), "--fake"], timeout=120)
        E14_RESULTS["nccl"] = json.loads(_o.read_text()) if _o.exists() else {"error": "fake bench wrote nothing"}
    if E14_STAGING is not None:
        E14_STATE.enter("STAGING_JOIN")
        E14_STAGING.join(timeout=max(0.0, SGL_GIVE_UP_AT - 20 * 60 - time.perf_counter()))
        res = E14_STAGING.result
        if res is None:
            E14_RESULTS["staging"] = {"still_running": True, "note": "launch from the original paths"}
        else:
            paths, rep = res
            E14_RESULTS["staging"] = rep
            SGL_MODEL_PATH = Path(paths.get(str(SGL_MODEL_PATH), str(SGL_MODEL_PATH)))
            SGL_DRAFT_PATH = Path(paths.get(str(SGL_DRAFT_PATH), str(SGL_DRAFT_PATH)))
        E14_RESULTS["launch_paths"] = {"model": str(SGL_MODEL_PATH), "draft": str(SGL_DRAFT_PATH)}
        e14_save()
        print("E14 staging:", json.dumps(E14_RESULTS["staging"])[:1200], flush=True)
    E14_STATE.enter("LAUNCH", deadline=SGL_GIVE_UP_AT)


E11_HARNESS_E14 = os.environ.get("E11_HARNESS") == "1"
E14_TS = float(os.environ.get("E14_TIME_SCALE", "1"))    # the CPU harness shortens every window and phase cap; 1 on Kaggle
_ts = lambda s: max(1.5, float(s) * E14_TS)
CONFIG["e14"] = {"measure": E14_DO_MEASURE, "staging": E14_STAGING is not None, "raw_reasoning": True,
                 "submit_arm": None if E14_DO_MEASURE else "DFlash width 6 + CUDA graphs 1..13 (Notes/30)",
                 "modules": _E14_MOD_SHA, "arc2_duck_sha": "758bddb5aeeb437d059f70a8de41451b4e00e18adde6a251b66d901ec53a40da"}
e14_save()


In [ ]:
# ==================================================================== CELL 2 — offline install (PIV/V3.1 parity block) + tokenizer
import importlib, importlib.metadata
def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}
_loaded = {"vllm": "vllm", "transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}
before_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(f"wheelhouse not attached: {WHEELHOUSE}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

# S1: read the model files into the page cache in the background while pip runs (read-only, best effort)
if os.environ.get("Q27X_SKIP_INSTALL") != "1" and E14_STAGING is None:   # E14: ranged staging replaces the cat
    try:
        import shlex as _shlex
        _wdirs = (SGL_MODEL_PATH, SGL_DRAFT_PATH) if ENGINE_BACKEND == "sglang" else (MODEL_PATH,)   # E11-05
        _wfiles = sorted(str(p) for _d in _wdirs for p in Path(_d).glob("*.safetensors"))
        if _wfiles:
            subprocess.Popen(["bash", "-c", "cat " + " ".join(_shlex.quote(p) for p in _wfiles) + " > /dev/null"],
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("weight prefetch started:", len(_wfiles), "files")
    except Exception as _e:
        print("weight prefetch not started:", repr(_e)[:200])

if os.environ.get("Q27X_SKIP_INSTALL") != "1":     # Q27X_SKIP_INSTALL is for the local harness only
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Python {sys.version.split()[0]}: the cp312 wheelhouse needs the pinned Python 3.12 Kaggle image")
    for _pkg, _mod in _loaded.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _mod in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the session before Run All.")
    _tok_pkgs = [f"{p}=={EXPECTED_RUNTIME_PACKAGES[p]}" for p in ("transformers", "tokenizers", "huggingface-hub")
                 if before_stack[p] != EXPECTED_RUNTIME_PACKAGES[p]]
    if ENGINE_BACKEND == "sglang" and before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        # E11-05: the kernel only tokenizes (S49 stack); SGLang runs in its own venv; vLLM is installed only for the fallback
        try:
            offline_pip_install(_tok_pkgs)
        except Exception as _e:
            print("tokenizer-only install failed -> the full S49 install:", repr(_e)[:300], flush=True)
            offline_pip_install(["vllm==0.19.0"])
    elif before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        offline_pip_install(_tok_pkgs)
    after_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}
    for _p, _want in EXPECTED_RUNTIME_PACKAGES.items():
        if _p == "vllm" and ENGINE_BACKEND == "sglang":
            continue                                   # E11-05: checked by the fallback cell, only if it runs
        if after_stack.get(_p) != _want:
            raise RuntimeError(f"runtime parity failure: {_p}={after_stack.get(_p)!r}, expected {_want!r}")
    print("runtime parity PASS:", after_stack)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
_model_cfg = _load_json(MODEL_PATH / "config.json")
assert _model_cfg.get("model_type") == "qwen3_5", _model_cfg.get("model_type")
assert (_model_cfg.get("quantization_config") or {}).get("quant_method") == "fp8"
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)
assert len(THINK_CLOSE_IDS) == 1, THINK_CLOSE_IDS
THINK_CLOSE_ID = THINK_CLOSE_IDS[0]
STOP_IDS = tokenizer.encode(STOP_THINKING_TEXT, add_special_tokens=False)
assert THINK_CLOSE_ID in STOP_IDS
_gen_cfg = _load_json(MODEL_PATH / "generation_config.json") if (MODEL_PATH / "generation_config.json").exists() else {}
_eos = _gen_cfg.get("eos_token_id", [])
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])     # v1.1 (F8): stripped before a forced close
print("tokenizer OK | </think> id", THINK_CLOSE_ID, "| stop text", len(STOP_IDS), "tokens")

# E11-05: the SGLang server receives token ids made by this (S49) tokenizer. Check that the INT4 checkpoint has the same
# vocabulary (with added tokens) and tokenizes a rendered prompt identically; otherwise run the S49 vLLM engine.
E11_TOKENIZER_CHECK = {"ok": None, "skipped": ENGINE_BACKEND != "sglang"}
if ENGINE_BACKEND == "sglang":
    try:
        from tokenizers import Tokenizer as _HFTokenizer
        _t4 = _HFTokenizer.from_file(str(SGL_MODEL_PATH / "tokenizer.json"))
        _probe = tokenizer.apply_chat_template(
            [{"role": "user", "content": "Example 1\nInput:\n0 1 2\n3 4 5\nOutput:\n5 4 3\n```python\ndef transform(grid):\n    return grid\n```"}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True, reasoning_effort=REASONING_EFFORT)
        _probe += "Let me think.\n</think>\n\n```python\nimport numpy as np\n```<|im_end|>"
        # rev 2: + a real ARC prompt fragment (every train pair of the first test task, rendered as digit rows)
        _t0 = test_challenges[sorted(test_challenges)[0]]
        _probe += "".join(f"\nExample {_i + 1}\nInput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["input"])
                          + "\nOutput:\n" + "\n".join(" ".join(str(v) for v in _row) for _row in _p["output"])
                          for _i, _p in enumerate(_t0["train"]))
        _ids_s49 = tokenizer.encode(_probe, add_special_tokens=False)
        _ids_int4 = _t4.encode(_probe, add_special_tokens=False).ids
        _same_vocab = _t4.get_vocab(with_added_tokens=True) == tokenizer.backend_tokenizer.get_vocab(with_added_tokens=True)
        # rev 2: the BPE merges too (the two files store them in different formats: "a b" strings vs [a, b] pairs)
        _norm = lambda m: [tuple(x.split(" ", 1)) if isinstance(x, str) else tuple(x) for x in m]
        _same_merges = (_norm(_load_json(SGL_MODEL_PATH / "tokenizer.json")["model"]["merges"])
                        == _norm(_load_json(MODEL_PATH / "tokenizer.json")["model"]["merges"]))
        _int4_gen = _load_json(SGL_MODEL_PATH / "generation_config.json") if (SGL_MODEL_PATH / "generation_config.json").exists() else {}
        _e4 = _int4_gen.get("eos_token_id", [])
        _same_eos = set(_e4 if isinstance(_e4, list) else [_e4]) == EOS_IDS
        E11_TOKENIZER_CHECK = {"ok": bool(_same_vocab and _same_merges and _ids_s49 == _ids_int4 and _same_eos),
                               "same_vocab": _same_vocab, "same_merges": _same_merges, "same_probe_ids": _ids_s49 == _ids_int4,
                               "probe_tokens": len(_ids_s49), "same_eos": _same_eos}
    except Exception as _e:
        E11_TOKENIZER_CHECK = {"ok": False, "error": f"{type(_e).__name__}: {_e}"[:300]}
    if not E11_TOKENIZER_CHECK["ok"]:
        print("E11 tokenizer check FAILED -> S49 vLLM FP8 engine:", E11_TOKENIZER_CHECK, flush=True)
        ENGINE_BACKEND = "vllm"
CONFIG["e11"]["tokenizer_check"] = E11_TOKENIZER_CHECK
print("E11 tokenizer check:", E11_TOKENIZER_CHECK)


In [ ]:
# ==================================================================== CELL 3 — E12: Duck package (verbatim) + ARC-2 adapter
# Tufa Labs' Duck harness as forked by Daniel Franzen (da-fr/arc-agi-3-solution @ 10882e332d, Apache-2.0,
# LICENSE + NOTICE inside). The 20 files of inference/ are byte-identical to that commit (sha256 manifest below);
# arc2_duck.py is the only new code (ARC-2 prompt, prelude with check/submit, host re-verification, driver, selection).
import base64 as _b64, io as _io, logging as _logging
E12_DIR = Path(os.environ.get("E12_DIR", "/tmp/e12_duck"))
_E12_ZIP_BYTES = q27x_payload("q27x_e12_duck_6f48b81af59a.bin", "6f48b81af59ad9e1d5d8a94092cc86d017f036d2710f33280838f0b33689c35b")   # Notes/21: payload dataset
_E12_MANIFEST = {
"arc2_duck.py": "758bddb5aeeb437d059f70a8de41451b4e00e18adde6a251b66d901ec53a40da",
"vendor/LICENSE": "3ddf9be5c28fe27dad143a5dc76eea25222ad1dd68934a047064e56ed2fa40c5",
"vendor/NOTICE.txt": "1e2b756570ed5a6d85d06e61e458021c1221a5f72ad33165ea29bb85b9053cf1",
"vendor/inference/__init__.py": "d92fed573b404870f7c6a1cef8f3e5d696195e2886e9e9183485c304fe5090f8",
"vendor/inference/agent/__init__.py": "8c561093771390b6f64fc0e87186982cd04045e8b6a2cf8d7f17e494762b4385",
"vendor/inference/agent/action_names.py": "8dd40e957046977e811eb2c2eb13fce905dbecf77f8dc7f846540f4d48af963d",
"vendor/inference/agent/noop_repeat_guard.py": "e9172d58687e9043c3445f87e11bdf74c4104d8b0bed5fb6b54745a6aa58655a",
"vendor/inference/agent/priority_pace_reference.json": "f29c16e336aef6b0dbf73ec056f554bd5192ac02f8deeda70ad152ef1214d0ab",
"vendor/inference/agent/priority_scheduler.py": "3a13659001458ea1a45ee3a812babe1b0012409eeabb23d681395796941b8e94",
"vendor/inference/agent/prompts.py": "14965a99bc55a61dad59fb79038f56c0ba1d1368350c8487b5a7b9edef0488ac",
"vendor/inference/agent/python_tool_sandbox.py": "c2208578220862aceb352138b9c3a3bb7347f1e6a4ad09d1d9cb3f2ee3507477",
"vendor/inference/agent/runtime_state.py": "b5c55d561a63e2dcf0422ee7fd339923a9fb909c779302bbf02b8ea49ccc607f",
"vendor/inference/agent/tool_agent.py": "485ef8a98308654b9dbb69535bc3c6ed3723c80bbcd5a4538e3ce181e6fda615",
"vendor/inference/agent/vision_context.py": "51d54a7aa9f1cd8e41b3fcc00ffd1a9cd1ada6fcde261d09faa49561fd843466",
"vendor/inference/utils/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
"vendor/inference/utils/animation.py": "6c14b613b3a7c66ce3373162d6faa1469a1ddb3ca4ab882299998c5cf14add96",
"vendor/inference/utils/frame_diff.py": "6ac3112773ffa6b1562b1e8f35a8018bd0dec9de7a78b12e7b13c529d4f46af8",
"vendor/inference/utils/grid_utils.py": "1e43809c9a4a463c65f897c48abb8e6cf4b28a32ac4ec32a07e9adcf06ab0c03",
"vendor/inference/utils/openai_compat.py": "fea0a75081372769fd854997b99e7826b334dcc5966876f31f587d19ae961165",
"vendor/inference/utils/retained_functions.py": "0d8b267cb387565c04d4977116020cc04d84ad9d9175d2bf4bc4613703cc5a01",
"vendor/inference/utils/run_artifacts.py": "789d541b395043379c23f5b0b24044e50fadab02e7f86f64cdd54714f4cab339",
"vendor/inference/utils/segmentation.py": "0d19669e5a36351e41a6b94b7c68f5b258829de78441a03a8749687c262b452c",
"vendor/inference/utils/viewer_artifacts.py": "cb5043dab8f5294608cda3675408ced904b849ae4985829b4ffd2d84b8e5d807"
}
if E12_DIR.exists():
    shutil.rmtree(E12_DIR)
with zipfile.ZipFile(_io.BytesIO(_E12_ZIP_BYTES)) as _z:
    _z.extractall(E12_DIR)
_bad = [p for p, h in _E12_MANIFEST.items() if hashlib.sha256((E12_DIR / p).read_bytes()).hexdigest() != h]
assert not _bad, ("E12 package hash mismatch", _bad)
sys.path[:0] = [str(E12_DIR), str(E12_DIR / "vendor")]
# E16: overlay ONLY arc2_duck.py (sha-pinned, compressed in this notebook) after the payload's sha check, before
# the import; the payload dataset is unchanged
import zlib as _e16_zlib
_e16_adapter = _e16_zlib.decompress(_b64.b64decode("..."))
assert hashlib.sha256(_e16_adapter).hexdigest() == "673618ed4cf381ef0022f2d6a844eddc4657877f63ed7246e8526a30ab0ee6b0", "E16 adapter sha"
(E12_DIR / "arc2_duck.py").write_bytes(_e16_adapter)
assert hashlib.sha256((E12_DIR / "arc2_duck.py").read_bytes()).hexdigest() == "673618ed4cf381ef0022f2d6a844eddc4657877f63ed7246e8526a30ab0ee6b0"
_E12_MANIFEST["arc2_duck.py"] = "673618ed4cf381ef0022f2d6a844eddc4657877f63ed7246e8526a30ab0ee6b0"
E14_RESULTS["build"]["arc2_duck_sha"] = "673618ed4cf381ef0022f2d6a844eddc4657877f63ed7246e8526a30ab0ee6b0"
E14_RESULTS["build"]["e16_adapter_overlay"] = True
print("E16 adapter OK:", "673618ed4cf3", flush=True)
import arc2_duck as E12A
# Duck reads its settings at import: set them first. The base URL is the engine cell's (same port); the stream count is
# set again from the engine's concurrency before the gate is first used (read at call time).
os.environ.update(E12A.duck_env(f"http://127.0.0.1:{int(os.environ.get('E11_PORT', 30000))}/v1", "pending", SGL_MAX_RUNNING))
_E12_TA, _E12_DP, _e12_run_sandboxed, _ = E12A._import_duck()
E12_SANDBOX_LIMIT = E12A.install_sandbox_memory_limit(E12_SANDBOX_MEM_MB)   # rev 2: every sandbox child, before its code


class _E12LevelCountFilter(_logging.Filter):   # Duck's per-game level-count warning has no ARC-2 meaning (priority uses 10)
    def filter(self, record):
        return "total level count unavailable" not in record.getMessage()


_logging.getLogger(_E12_TA.__name__).addFilter(_E12LevelCountFilter())

# sandbox self-test with our own scripted code (the engine cell runs only if it passes)
_t = time.perf_counter()
try:
    _st_task = test_challenges[sorted(test_challenges)[0]]
    _st = _e12_run_sandboxed(
        code=E12A.task_prelude(_st_task) + "def _e12_ident(g):\n    return g\nres = check(_e12_ident, verbose=False)\nprint(len(res), sum(1 for r in res if 'ok' in r))",
        timeout_seconds=30, initial_state={"current_frame": None, "history": [], "valid_actions": [], "last_action_call_result": {}},
        action_handler=lambda a, stale_after=None: {"action_result": {}, "state": {}}, kept_functions=[], retain_imports=True)
    _n = len(_st_task["train"])
    E10_SANDBOX_SELFTEST = {"ok": (_st.get("stdout") or "").strip() == f"{_n} {_n}" and not _st.get("error"),
                            "stdout": (_st.get("stdout") or "")[:200], "error": (_st.get("error") or "")[:500],
                            "sec": round(time.perf_counter() - _t, 2), "python": sys.version.split()[0]}
except Exception as _e:
    E10_SANDBOX_SELFTEST = {"ok": False, "error": f"{type(_e).__name__}: {_e}"[:500]}
# rev 2: memory limit evidence (reported; the run continues without it rather than not at all). The 5 GiB request is
# made only after the read-back confirmed the limit, so it can never really allocate.
_mem = {"limit": dict(E12_SANDBOX_LIMIT)}
try:
    _st_kw = dict(timeout_seconds=30, initial_state={"current_frame": None, "history": [], "valid_actions": [],
                                                     "last_action_call_result": {}},
                  action_handler=lambda a, stale_after=None: {"action_result": {}, "state": {}}, kept_functions=[],
                  retain_imports=True)
    if E12_SANDBOX_LIMIT.get("readback_ok"):
        _big = _e12_run_sandboxed(code="x = bytearray(5 * 1024 ** 3)\nprint(len(x))", **_st_kw)
        _small = _e12_run_sandboxed(code="x = bytearray(128 * 1024 ** 2)\nprint(len(x))", **_st_kw)
        _mem.update(over_limit_rejected="MemoryError" in (_big.get("error") or ""),
                    small_ok=(_small.get("stdout") or "").strip() == str(128 * 1024 ** 2))
        _mem["ok"] = bool(_mem["over_limit_rejected"] and _mem["small_ok"])
    else:
        _mem["ok"] = False
except Exception as _e:
    _mem.update(ok=False, error=f"{type(_e).__name__}: {_e}"[:300])
_mem["limit"] = dict(E12_SANDBOX_LIMIT)
E10_SANDBOX_SELFTEST["memory_limit"] = _mem
if E12_SANDBOX_MEM_MB > 0 and not _mem.get("ok"):
    print("E12 WARNING: sandbox memory limit NOT verified:", json.dumps(_mem), flush=True)
    if os.name == "posix":   # rev 3 (GPT re-audit): on Kaggle (Linux) never run model code without the limit
        E10_SANDBOX_SELFTEST["ok"] = False
        E10_SANDBOX_SELFTEST["error"] = "memory limit not verified (fail closed): " + json.dumps(_mem)[:300]
        print("E12: solver disabled (fail closed); CELL 0 fallback submission stays", flush=True)
CONFIG["e12"]["sandbox_selftest"] = E10_SANDBOX_SELFTEST
if _E12_REV5:   # rev 5 R5-4: is B3 counting reasoning with the tokenizer (else a fallback ratio)? recorded, never fatal
    CONFIG["e12"]["rev5_tokenizer"] = E12A.tokenizer_status()
    print("E12 rev 5 reasoning tokenizer:", json.dumps(CONFIG["e12"]["rev5_tokenizer"]), flush=True)
print("E12 package OK:", len(_E12_MANIFEST), "files | sandbox self-test:", json.dumps(E10_SANDBOX_SELFTEST), flush=True)


In [ ]:
# E10: original task-order and evaluation profiles, but one Poetiq program per task/expert/iteration.
if P['source']=='eval':
    _ch=_load_json(ARC_ROOT/'arc-agi_evaluation_challenges.json')
    _sol=_load_json(ARC_ROOT/'arc-agi_evaluation_solutions.json')
    if P.get('task_list'):
        assert all(t in _ch for t in P['task_list']), 'Evaluation task ID missing'
        TASK_IDS=sorted(P['task_list'])
    else:
        TASK_IDS=sorted(random.Random(TASK_SAMPLE_SEED).sample(sorted(_ch),P['n_tasks']))
    CHALLENGES={t:_ch[t] for t in TASK_IDS}
    SOLUTIONS={t:_sol[t] for t in TASK_IDS}
else:
    CHALLENGES=dict(test_challenges)
    TASK_IDS=sorted(CHALLENGES) # no Codex ordering
    SOLUTIONS=None
N_HIDDEN_UNITS=sum(len(t['test']) for t in test_challenges.values())
print('E10 Poetiq:',len(TASK_IDS),'tasks,',sum(len(CHALLENGES[t]['test']) for t in TASK_IDS),
      'queries | experts',POETIQ_NUM_EXPERTS,'iterations per expert',POETIQ_MAX_ITERATIONS)


In [ ]:
# E12: = E11-06 CELL 5 unchanged except the Duck serving flags in e11_flags().
# ==================================================================== CELL 5 — engine: SGLang 0.5.21 + RedHat INT4 + DFlash2 (E11-04 W1)
# E11-05. The scheduler (next cells) is S49's, unchanged. It drives the engine with the vLLM calls it already uses
# (add_request / step / abort_request, DELTA token ids, finish 'stop' | 'length' | 'abort'); the shim below maps them onto
# SGLang's HTTP API: /generate streamed with --incremental-streaming-output (disjoint deltas, tokenizer_manager.py:2493-2515),
# /abort_request (the stream then ends with finish type 'abort'), /metrics. Launch = E11-04 W1, measured 2026-10-18.
# If SGLang does not come up (inputs, install, launch, smoke incl. abort), the server is killed and CELL 5b runs S49's vLLM.
import queue as _queue, signal as _signal, statistics as _statistics, types as _types, urllib.request as _urlreq
E11_HARNESS = os.environ.get("E11_HARNESS") == "1"            # local CPU harness only: fake server, no install, no GPU
SGL_PORT = int(os.environ.get("E11_PORT", 30000))
SGL_URL = f"http://127.0.0.1:{SGL_PORT}"
E11_SGL = {"requested": ENGINE_BACKEND, "ok": False, "stage": "not started", "inputs": {}, "install": None, "attempts": [],
           "caps": None, "launch_flags": None, "smoke": None, "error": None, "tokenizer_check": E11_TOKENIZER_CHECK}
E11_SERVER = None
ENGINE_READY = False; llm = None; engine = None; CONCURRENCY = 0
ENGINE_INFO = {"backend": None, "sglang": E11_SGL, "kv_tokens": None, "concurrency": None, "fatal": None, "load_sec": None}


class SamplingParams:
    """vLLM-shaped container. The shim sends max_tokens, temperature, top_p, top_k. The seed is not sent: SGLang 0.5.21
    uses sampling_seed only under --enable-deterministic-inference (sampling_batch_info.py:143-157), so sampling is random."""
    def __init__(self, max_tokens=16, seed=None, temperature=1.0, top_p=1.0, top_k=-1, **kw):
        self.max_tokens, self.seed = int(max_tokens), seed
        self.temperature, self.top_p, self.top_k = float(temperature), float(top_p), int(top_k)
        self.extra = kw


class RequestOutputKind:
    DELTA = "delta"


def make_params(max_tokens, seed):
    return SamplingParams(max_tokens=int(max_tokens), seed=int(seed), detokenize=False, skip_special_tokens=False,
                          output_kind=RequestOutputKind.DELTA, **SAMPLING)


class E11Fail(Exception):
    pass


def _e11_post(path, payload, timeout=30):
    req = _urlreq.Request(SGL_URL + path, data=json.dumps(payload).encode(), method="POST",
                          headers={"Content-Type": "application/json"})
    with _urlreq.urlopen(req, timeout=timeout) as r:
        return r.read()


def _e11_gpu_used():
    try:
        return [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                               capture_output=True, text=True, timeout=30).stdout.split()]
    except Exception:
        return []


def _e11_gpu_free(max_sec=180):
    t = time.perf_counter()
    while time.perf_counter() - t < max_sec:
        u = _e11_gpu_used()
        if E11_HARNESS or (u and max(u) < 1024):
            return {"free": True, "sec": round(time.perf_counter() - t), "used_mb": u}
        time.sleep(5)
    for _pat in ("sglang.launch_server", "sglang::"):
        subprocess.run(["pkill", "-9", "-f", _pat], capture_output=True)
    time.sleep(15)
    return {"free": False, "sec": round(time.perf_counter() - t), "used_mb": _e11_gpu_used(), "pkill": True}


# ---- install (= E11-03/04 C1): a venv from the wheelhouse lock, --no-deps; CUDA 13 pip runtime as CUDA_HOME; link shims
def e11_install():
    if E11_HARNESS:
        return Path(sys.executable), dict(os.environ), {"harness": True}
    shims, t0 = {}, time.perf_counter()
    venv = Path("/tmp/sglang-venv"); vpy = venv / "bin" / "python"
    # Kaggle drops '+' from dataset file names (flashinfer_jit_cache-0.6.18+cu130 -> ...0.6.18cu130); rebuild the original
    # names from SHA256SUMS as links in a find-links folder, checking the hash of every renamed wheel
    fl = Path("/tmp/e11-wheels"); fl.mkdir(parents=True, exist_ok=True)
    renamed = []
    for line in (SGL_WHEELHOUSE / "SHA256SUMS").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        h, name = line.split("  ", 1); name = name.strip()
        if not name.endswith(".whl"):
            continue
        s = SGL_WHEELHOUSE / name if (SGL_WHEELHOUSE / name).exists() else SGL_WHEELHOUSE / name.replace("+", "")
        if not s.exists():
            raise E11Fail(f"wheel missing from the dataset: {name}")
        if s.name != name:
            d = hashlib.sha256()
            with open(s, "rb") as f:
                for b in iter(lambda: f.read(16 * 1024 * 1024), b""):
                    d.update(b)
            if d.hexdigest() != h:
                raise E11Fail(f"sha256 mismatch for {s.name}")
            renamed.append(f"{s.name} -> {name}")
        if not (fl / name).exists():
            (fl / name).symlink_to(s)
    shims["renamed_wheels"] = renamed
    if not vpy.exists():
        subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv)], check=True)
        r = subprocess.run([sys.executable, "-m", "pip", "--python", str(vpy), "install", "--no-index", "--find-links", str(fl),
                            "--requirement", str(SGL_WHEELHOUSE / "requirements.lock"), "--no-deps", "--only-binary", ":all:",
                            "--no-compile", "--disable-pip-version-check", "--no-warn-conflicts", "-q"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise E11Fail("offline SGLang install failed: " + (r.stderr or r.stdout)[-1500:])
    site = Path(subprocess.run([str(vpy), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                               capture_output=True, text=True, check=True).stdout.strip())
    cuda_home = site / "nvidia" / "cu13"
    # E11-02 BUILD_PROBE: the pip CUDA runtime ships lib/libcudart.so.13 only; JIT links need libcudart.so and lib64
    if not (cuda_home / "lib" / "libcudart.so").exists():
        (cuda_home / "lib" / "libcudart.so").symlink_to("libcudart.so.13"); shims["libcudart.so"] = "created"
    if not (cuda_home / "lib64").exists():
        (cuda_home / "lib64").symlink_to("lib"); shims["lib64"] = "created"
    # E10 run 1: FlashInfer's JIT link needs -lcuda; Kaggle has only libcuda.so.1
    libcuda = Path("/tmp/e11-libcuda")
    have = [d for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs", "/usr/lib/x86_64-linux-gnu", "/usr/lib64")
            if Path(d, "libcuda.so").exists()]
    if have:
        shims["libcuda.so"] = f"exists in {have[0]}"
    else:
        real = None
        try:
            for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                if "libcuda.so.1" in line and "=>" in line:
                    real = line.split("=>")[-1].strip(); break
        except Exception:
            pass
        for d in ("/usr/local/nvidia/lib64", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
            if real is None and Path(d, "libcuda.so.1").exists():
                real = str(Path(d, "libcuda.so.1"))
        if real:
            libcuda.mkdir(parents=True, exist_ok=True)
            if (libcuda / "libcuda.so").is_symlink() or (libcuda / "libcuda.so").exists():
                (libcuda / "libcuda.so").unlink()
            (libcuda / "libcuda.so").symlink_to(Path(real).resolve()); shims["libcuda.so"] = f"shim -> {real}"
        else:
            shims["libcuda.so"] = "libcuda.so.1 NOT FOUND"
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_HOME", "CUDA_PATH")}
    env.update({
        "PATH": os.pathsep.join([str(venv / "bin"), str(cuda_home / "bin"), os.environ.get("PATH", "")]),
        "CUDA_HOME": str(cuda_home), "CUDA_PATH": str(cuda_home),
        "LIBRARY_PATH": os.pathsep.join(x for x in (str(libcuda), str(cuda_home / "lib"), os.environ.get("LIBRARY_PATH", "")) if x),
        "LD_LIBRARY_PATH": os.pathsep.join(x for x in (str(cuda_home / "lib"), os.environ.get("LD_LIBRARY_PATH", "")) if x),
        "TORCH_CUDA_ARCH_LIST": "8.9", "MAX_JOBS": "4",
        "SGLANG_CACHE_DIR": "/tmp/e11-cache/sglang", "CUDA_CACHE_PATH": "/tmp/e11-cache/cuda",
        "TRITON_CACHE_DIR": "/tmp/e11-cache/triton", "TORCH_EXTENSIONS_DIR": "/tmp/e11-cache/torch_ext",
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",
        "CUDA_DEVICE_ORDER": "PCI_BUS_ID", "PYTHONUNBUFFERED": "1",
    })
    env.update(E14L.jit_cache_env())      # E14: SGLANG_JIT_CACHE_DIR set explicitly (it is not under SGLANG_CACHE_DIR)
    shims.update(cuda_home=str(cuda_home), venv_python=str(vpy), sec=round(time.perf_counter() - t0, 1))
    return vpy, env, shims


# ---- launch: the E11-04 W1 command; one diagnosed fallback per cause (E11-04 rev 2b next_launch)
W1 = {"strategy": "extra_buffer_lazy", "mamba_cache": 56, "no_prefill_graph": True, "mem": "0.83"}
E11_OOM_MARKERS = ("out of memory", "outofmemory", "cuda error: out of memory", "not enough memory", "insufficient memory",
                   "cudaerrormemoryallocation")


def e11_flags(cfg):
    f = ["--model-path", str(SGL_MODEL_PATH), "--tp-size", "4", "--kv-cache-dtype", "fp8_e4m3",
         "--context-length", str(SGL_CONTEXT_LEN), "--max-running-requests", str(SGL_SERVER_MAX_RUNNING),
         "--chunked-prefill-size", "4096", "--mamba-ssm-dtype", "float16", "--enable-metrics", "--disable-flashinfer-autotune",
         "--random-seed", str(SEED), "--host", "127.0.0.1", "--port", str(SGL_PORT), "--log-level", "info",
         "--cuda-graph-max-bs-decode", str(SGL_MAX_RUNNING), "--watchdog-timeout", "1800", "--decode-log-interval", "40",
         "--speculative-algorithm", "DFLASH", "--speculative-draft-model-path", str(SGL_DRAFT_PATH),
         "--speculative-draft-model-quantization", "unquant", "--speculative-num-draft-tokens", "8"]
    if cfg["no_prefill_graph"]:
        f += ["--disable-prefill-cuda-graph"]
    f += ["--mamba-radix-cache-strategy", cfg["strategy"], "--max-mamba-cache-size", str(cfg["mamba_cache"]),
          "--mem-fraction-static", cfg["mem"],
          # E11-05: streaming for the shim (disjoint deltas). rev 2: interval 1 = a chunk every decode step. With N > 1 the
          # rule is len(output_ids) % N == 1 (output_streamer.py:427-436); DFLASH accepts several tokens per step, so a
          # stream could skip that residue for long stretches and the drain would lose the unsent thinking.
          "--incremental-streaming-output", "--stream-interval", "1"]
    # E12: Duck milestone-2 serving flags (dfranzen notebook cell 12; each verified in SGLang 0.5.21): OpenAI tool calls
    # and reasoning split for /v1/chat/completions, preserved thinking by default, prefix-aware scheduling, cached_tokens
    # in usage. /generate (the smoke below) is unaffected.
    f += ["--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
          "--default-chat-template-kwargs", json.dumps({"preserve_thinking": True}),
          "--schedule-policy", "lpm", "--enable-cache-report"]
    if E12_MAMBA_TRACK_64:
        f += ["--mamba-track-interval", "64", "--page-size", "64"]
    return f


# E14S (Notes/30, R2): the adopted arm = DFlash width 6 (531 vs 505 tok/s, gate PASS) on CUDA graphs 1..13 (captured
# 12 and 13, gate PASS). Set in every launch unless the E14A measurements run (they start from W1 and set each arm per
# launch).
E14_SUBMIT_ARM = {"--cuda-graph-bs-decode": [str(b) for b in range(1, SGL_SERVER_MAX_RUNNING + 1)],
                  "--speculative-num-draft-tokens": ["6"]}
E14_SUBMIT_DROP = ("--cuda-graph-max-bs-decode",)
_e11_flags_w1 = e11_flags


def e11_flags(cfg):
    f = _e11_flags_w1(cfg)
    if E14_DO_MEASURE:
        return f
    out, i, names = [], 0, set(E14_SUBMIT_ARM) | set(E14_SUBMIT_DROP)
    while i < len(f):
        if f[i] in names:
            i += 1
            while i < len(f) and not str(f[i]).startswith("--"):
                i += 1
            continue
        out.append(f[i])
        i += 1
    for k, v in E14_SUBMIT_ARM.items():
        out += [k] + list(v)
    return out


def e11_next(cfg, tail, used):
    """(cfg, reason) for the one diagnosed relaunch of this cause, or None. Lazy / prefill causes are read from error
    lines only (normal startup lines also say 'Mamba Cache ...')."""
    t = "\n".join(l for l in (tail or "").splitlines()
                  if any(k in l.lower() for k in ("error", "exception", "assert", "unsupported", "not supported")))[-4000:].lower()
    if any(k in (tail or "").lower() for k in E11_OOM_MARKERS) and cfg["mem"] == "0.83" and "oom" not in used:
        return dict(cfg, mem="0.78"), "oom"
    if cfg["strategy"] == "extra_buffer_lazy" and ("lazy" in t or "extra_buffer" in t or "mamba" in t) and "lazy" not in used:
        return dict(cfg, strategy="extra_buffer", mamba_cache=70), "lazy"          # 5 slots per request -> 14 >= 12
    if cfg["no_prefill_graph"] and "graph" in t and "prefill" not in used:
        return dict(cfg, no_prefill_graph=False), "prefill"
    return None


def e11_slots(cfg):
    """kv_cache_configurator.py:2232-2261 with the overlap scheduler on: extra_buffer 5, extra_buffer_lazy 4 slots."""
    return {"extra_buffer_lazy": 4, "no_buffer": 3}.get(cfg["strategy"], 5)


def e11_caps(text):
    """Effective limits from the server log (/get_server_info reports the requested values)."""
    caps = {}
    m = re.findall(r"max_running_requests is capped to (\d+) by the mamba state cache \(max_mamba_cache_size=(\d+), (\d+) state", text)
    caps["capped_to"] = min(int(a) for a, _, _ in m) if m else None
    tp0 = "\n".join(l for l in text.splitlines() if "TP0]" in l) or text
    kv = re.findall(r"KV Cache is allocated\. dtype: ([\w.]+), #tokens: (\d+), K size: ([\d.]+) GB, V size: ([\d.]+) GB", tp0)
    caps["kv_tokens"] = int(kv[0][1]) if kv else None                     # first TP0 line = target (second = drafter)
    mc = re.findall(r"Mamba Cache is allocated\. max_mamba_cache_size: (\d+)", tp0)
    caps["mamba_cache_size"] = int(mc[0]) if mc else None
    vb = re.findall(r"Capture target verify CUDA graph begin\..*?bs=\[([\d, ]+)\]", text)
    caps["verify_graph_bs"] = [int(x) for x in re.findall(r"\d+", vb[0])] if vb else None
    caps["prefill_graph_captured"] = "Capture target prefill CUDA graph" in text
    am = re.findall(r"avail mem=([\d.]+) GB", tp0)
    caps["avail_mem_gb_last"] = float(am[-1]) if am else None
    return caps


def e11_capacity_gate(caps, cfg):
    """rev 2 (GPT audit): fail-closed. Every limit must be in the server log; concurrency = the smallest of 12, the mamba
    cache // state slots (and the logged cap), the largest captured verify-graph batch <= 12, and KV // 70k."""
    slots = e11_slots(cfg)
    mc, kv, free = caps.get("mamba_cache_size"), caps.get("kv_tokens"), caps.get("avail_mem_gb_last")
    eff = min(SGL_MAX_RUNNING, mc // slots) if isinstance(mc, int) else None
    if eff is not None and caps.get("capped_to"):
        eff = min(eff, caps["capped_to"])
    bs = [b for b in (caps.get("verify_graph_bs") or []) if b <= SGL_MAX_RUNNING]
    caps.update(slots_per_request=slots, effective_running=eff)
    conc = min(SGL_MAX_RUNNING, eff or 0, max(bs) if bs else 0, (kv or 0) // SGL_KV_PER_REQUEST)
    g = {"mamba_cache_logged": eff is not None, "kv_logged": kv is not None, "verify_graph_logged": bool(bs),
         "running_min": (eff or 0) >= SGL_MIN_RUNNING, "kv_min": (kv or 0) >= SGL_MIN_KV,
         "prefill_graph_as_requested": (caps.get("prefill_graph_captured") is False) if cfg["no_prefill_graph"] else True,
         "headroom": isinstance(free, (int, float)) and free >= SGL_MIN_FREE_GB, "concurrency_min": conc >= SGL_MIN_RUNNING}
    g["pass"] = all(g.values())
    return g, conc


class E11Server:
    def __init__(self, cmd, env, tag):
        self.cmd, self.env, self.tag, self.proc, self.log_f = cmd, env, tag, None, None
        self.log_path = OUT / f"sglang_server_{tag}.log"

    def start(self):
        self.log_f = open(self.log_path, "w", encoding="utf-8")
        self.t0 = time.perf_counter()
        self.proc = subprocess.Popen(self.cmd, env=self.env, stdout=self.log_f, stderr=subprocess.STDOUT,
                                     cwd=None if E11_HARNESS else "/tmp", start_new_session=True)

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def log_text(self):
        try:
            return self.log_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    def wait_ready(self, timeout):
        # E14-0 (Notes/07, 11): stall-based. Progress = log growth or the server tree's read() bytes + page faults (the
        # mmap weight load is silent in the log); stalls: 900 s while loading, 720 s while compiling (> the ~6 min silent
        # first verify-graph capture); a progressing server is never given up before t0 + timeout
        def _health():
            _urlreq.urlopen(SGL_URL + "/health", timeout=10).read()
            return True
        mon = E14L.StallMonitor(self.proc, self.log_path, poll_sec=2 if E11_HARNESS else 5, log=print)
        ok, why, self.e14_stats = mon.wait(_health, give_up_at=self.t0 + timeout)
        return ok, why

    def kill(self, wait=True):
        if self.alive():
            for sig, w in ((_signal.SIGTERM, 30 if wait else 0), (getattr(_signal, "SIGKILL", _signal.SIGTERM), 30)):
                try:
                    if hasattr(os, "killpg"):
                        os.killpg(self.proc.pid, sig)          # the whole group: TP workers, scheduler, detokenizer
                    else:
                        self.proc.kill()
                except Exception:
                    pass
                try:
                    self.proc.wait(timeout=w if w else 0.1)
                    break
                except Exception:
                    continue
        if self.log_f:
            try:
                self.log_f.close()
            except Exception:
                pass
            self.log_f = None


def e11_kill_server(wait=True):
    if E11_SERVER is not None:
        E11_SERVER.kill(wait=wait)


# ---- the shim: the vLLM engine calls of the S49 scheduler on top of SGLang's HTTP API
class _E11Completion:
    __slots__ = ("token_ids", "finish_reason")

    def __init__(self, ids, reason):
        self.token_ids, self.finish_reason = ids, reason


class _E11Output:
    __slots__ = ("request_id", "outputs", "finished")

    def __init__(self, rid, ids, reason):
        self.request_id, self.outputs, self.finished = rid, [_E11Completion(ids, reason)], reason is not None


class E11SglangEngine:
    """add_request starts one streaming /generate per request in a thread; step() returns the token deltas that arrived
    (merged per request, order kept) and raises when the server died; abort_request posts /abort_request per id."""

    def __init__(self, server):
        self.server, self.q, self.aborted, self.done = server, _queue.Queue(), set(), {}
        self.lock = threading.Lock()
        self.stats = {"added": 0, "finished": 0, "by_reason": {}, "errors": [], "prompt_tokens": 0, "cached_tokens": 0,
                      "completion_tokens": 0, "aborts_sent": 0, "abort_errors": 0, "abort_timeouts": 0,
                      "abort_wait_sec_max": 0.0, "accept_len_sum": 0.0, "accept_len_n": 0}
        self.req_log = OUT / "sglang_requests.jsonl"

    def add_request(self, rid, prompt, params):
        if not self.server.alive():
            raise RuntimeError(f"SGLang server is not running (rc={self.server.proc.returncode})")
        sp = {"max_new_tokens": int(params.max_tokens), "temperature": float(params.temperature), "top_p": float(params.top_p),
              "top_k": int(params.top_k), "skip_special_tokens": False, "no_stop_trim": True}
        body = json.dumps({"rid": rid, "input_ids": [int(x) for x in prompt["prompt_token_ids"]],
                           "sampling_params": sp, "stream": True}).encode()
        with self.lock:
            self.stats["added"] += 1
        threading.Thread(target=self._stream, args=(rid, body, time.perf_counter()), daemon=True).start()

    def _stream(self, rid, body, t0):
        reason, meta, err, n = None, {}, None, 0
        try:
            req = _urlreq.Request(SGL_URL + "/generate", data=body, method="POST", headers={"Content-Type": "application/json"})
            with _urlreq.urlopen(req, timeout=SGL_STREAM_IDLE_SEC) as r:
                for raw in r:
                    line = raw.strip()
                    if not line.startswith(b"data:"):
                        continue
                    data = line[5:].strip()
                    if data == b"[DONE]":
                        break
                    d = json.loads(data)
                    if "error" in d:
                        err = str(d["error"])[:300]
                        break
                    ids = d.get("output_ids") or []
                    m = d.get("meta_info") or {}
                    fr = m.get("finish_reason")
                    if fr is not None and reason is None:
                        reason = (fr.get("type") if isinstance(fr, dict) else str(fr)) or "stop"
                        meta = m
                    n += len(ids)
                    if ids or reason is not None:
                        self.q.put((rid, list(ids), reason))
                    if reason is not None:
                        break
        except Exception as e:
            err = f"{type(e).__name__}: {e}"[:300]
        if reason is None:
            reason = "error"
            self.q.put((rid, [], "error"))
        rec = {"rid": rid, "finish": reason, "tokens": n, "sec": round(time.perf_counter() - t0, 1),
               "prompt_tokens": meta.get("prompt_tokens"), "cached_tokens": meta.get("cached_tokens"),
               "completion_tokens": meta.get("completion_tokens"), "spec_accept_length": meta.get("spec_accept_length"),
               "error": err}
        with self.lock:
            self.done[rid] = reason                     # rev 2: the stream has ended (abort_request waits for this)
            s = self.stats
            s["finished"] += 1; s["by_reason"][reason] = s["by_reason"].get(reason, 0) + 1
            s["prompt_tokens"] += int(meta.get("prompt_tokens") or 0); s["cached_tokens"] += int(meta.get("cached_tokens") or 0)
            s["completion_tokens"] += int(meta.get("completion_tokens") or 0)
            if meta.get("spec_accept_length") is not None:
                s["accept_len_sum"] += float(meta["spec_accept_length"]); s["accept_len_n"] += 1
            if err and len(s["errors"]) < 50:
                s["errors"].append(f"{rid}: {err}")
            try:
                with open(self.req_log, "a", encoding="utf-8") as f:
                    f.write(json.dumps(rec) + "\n")
            except Exception:
                pass

    def step(self):
        if not self.server.alive():
            raise RuntimeError(f"SGLang server exited (rc={self.server.proc.returncode}); see {self.server.log_path}")
        items = []
        try:
            items.append(self.q.get(timeout=0.05))
        except _queue.Empty:
            return []
        while True:
            try:
                items.append(self.q.get_nowait())
            except _queue.Empty:
                break
        merged = {}
        for rid, ids, reason in items:
            m = merged.setdefault(rid, [[], None])
            if m[1] is None:
                m[0].extend(ids); m[1] = reason
        return [_E11Output(rid, ids, reason) for rid, (ids, reason) in merged.items()]

    def abort_request(self, ids, wait_sec=30.0):
        """rev 2 (GPT audit, verified: /abort_request only dispatches; scheduler.py:5558-5572 finishes a running request
        after one more decode pass). All POSTs go out in parallel, then this waits (bounded, one shared deadline) until
        each stream has ended, so the slot is released before the caller admits phase 2. Returns rid -> final reason,
        'timeout' or the POST error."""
        ids = list(ids)
        if not ids:
            return {}
        t0, posted = time.perf_counter(), {}
        for rid in ids:
            self.aborted.add(rid)

        def _post(rid):
            try:
                _e11_post("/abort_request", {"rid": rid}, timeout=wait_sec)
                posted[rid] = True
            except Exception as e:
                posted[rid] = f"{type(e).__name__}: {e}"[:200]
        ths = [threading.Thread(target=_post, args=(rid,), daemon=True) for rid in ids]
        for t in ths:
            t.start()
        for t in ths:
            t.join(max(0.0, wait_sec - (time.perf_counter() - t0)))
        while time.perf_counter() - t0 < wait_sec and any(r not in self.done for r in ids):
            time.sleep(0.02)
        out = {}
        with self.lock:
            for rid in ids:
                p = posted.get(rid, "post timeout")
                if p is True:
                    self.stats["aborts_sent"] += 1
                else:
                    self.stats["abort_errors"] += 1
                    if len(self.stats["errors"]) < 50:
                        self.stats["errors"].append(f"abort {rid}: {p}")
                if rid in self.done:
                    out[rid] = self.done[rid]
                else:
                    out[rid] = "timeout" if p is True else p
                    self.stats["abort_timeouts"] += 1
            self.stats["abort_wait_sec_max"] = max(self.stats["abort_wait_sec_max"], round(time.perf_counter() - t0, 2))
        return out


_E11_PROM_RE = re.compile(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\{[^}]*\})?\s+(\S+)")


def e11_prom(timeout=10):
    txt = _urlreq.urlopen(SGL_URL + "/metrics", timeout=timeout).read().decode("utf-8", "replace")
    rows = []
    for line in txt.splitlines():
        if not line or line.startswith("#"):
            continue
        m = _E11_PROM_RE.match(line)
        if not m:
            continue
        try:
            rows.append(_types.SimpleNamespace(name=m.group(1), labels=m.group(2), value=float(m.group(3))))
        except ValueError:
            continue
    return rows


class E11SglangLLM:
    """llm.get_metrics() for the S49 KV gate: the gate reads names containing 'kv_cache_usage'. SGLang's full_token_usage
    (KV pool used, evictable cache excluded) is given that name; token_usage = max(full, mamba) is kept as it is."""

    def get_metrics(self):
        rows = e11_prom()
        full = [r.value for r in rows if r.name == "sglang:full_token_usage"]
        anyu = [r.value for r in rows if r.name == "sglang:token_usage"]
        kv = max(full) if full else (max(anyu) if anyu else None)
        if kv is not None:
            rows.append(_types.SimpleNamespace(name="e11:kv_cache_usage(sglang:full_token_usage)", labels=None, value=kv))
        return rows


_E11_SNAP_KEYS = ("num_running_reqs", "num_queue_reqs", "token_usage", "gen_throughput", "spec_accept", "cache_hit_rate",
                  "retract", "generation_tokens_total", "prompt_tokens_total", "cached_tokens_total", "kv_cache_usage",
                  "num_requests_total")


def metrics_snapshot():
    if not ENGINE_READY:
        return []
    try:
        return [{"name": r.name, "labels": r.labels, "value": r.value} for r in llm.get_metrics()
                if any(k in r.name for k in _E11_SNAP_KEYS) and not r.name.endswith(("_bucket", "_created"))]
    except Exception as e:
        return [{"error": repr(e)[:300]}]


_E11_KV_RE = re.compile(r"([A-Za-z#][A-Za-z#_ ()/\-]*?):\s*(-?\d+(?:\.\d+)?)")


def e11_decode_summary():
    """Decode rate by running level and by mean context per running request, from the server's TP0 decode lines."""
    if E11_SERVER is None:
        return None
    rows = []
    for line in E11_SERVER.log_text().splitlines():
        if "Decode batch" in line and "TP0]" in line:
            rows.append({k.strip(): float(v) for k, v in _E11_KV_RE.findall(line.split("Decode batch", 1)[1])})
    def med(v):
        return round(_statistics.median(v), 1) if v else None
    bins = {}
    for r in rows:
        n, ft, g = r.get("#running-req"), r.get("#full token"), r.get("gen throughput (token/s)")
        if not n or ft is None or g is None:
            continue
        ctx = ft / n
        k = "<10k" if ctx < 10000 else "10-20k" if ctx < 20000 else "20-40k" if ctx < 40000 else "40-60k" if ctx < 60000 else ">=60k"
        bins.setdefault(f"running={int(n)}" if n < SGL_MAX_RUNNING else "running=max", {}).setdefault(k, []).append(g)
    out = {"n_lines": len(rows),
           "running_hist": {str(int(k)): sum(1 for r in rows if r.get("#running-req") == k) for k in sorted({r.get("#running-req", 0) for r in rows})},
           "gen_tok_s_median_at_max": med([r["gen throughput (token/s)"] for r in rows
                                           if r.get("#running-req") == SGL_MAX_RUNNING and "gen throughput (token/s)" in r]),
           "accept_len_median": med([r["accept len"] for r in rows if "accept len" in r]),
           "queue_max": max((r.get("#queue-req", 0) for r in rows), default=None),
           "retract_lines": sum(1 for l in E11_SERVER.log_text().splitlines() if "retract" in l.lower()),
           "by_context": {lvl: {k: {"lines": len(v), "gen_tok_s_median": med(v)} for k, v in d.items()} for lvl, d in bins.items()}}
    try:
        write_json_atomic(OUT / "sglang_decode_summary.json", out)
    except Exception:
        pass
    return out


def e11_smoke(eng):
    """The shim's own path: (a) greedy 16 tokens, streamed deltas == completion_tokens; (b) abort mid-stream -> the stream
    ends with finish 'abort' (the drain and the hard stop rely on it to free the slot)."""
    res, t0 = {}, time.perf_counter()
    ids = tokenizer.encode("Reply with the digits 1 2 3 4 5 6 7 8:", add_special_tokens=False)
    eng.add_request("e11-smoke-a", {"prompt_token_ids": ids}, SamplingParams(max_tokens=16, temperature=0.0, top_p=1.0, top_k=-1))
    got, fin = [], None
    while fin is None and time.perf_counter() - t0 < 300:
        for o in eng.step():
            if o.request_id == "e11-smoke-a":
                got += o.outputs[0].token_ids; fin = o.outputs[0].finish_reason if o.finished else None
    time.sleep(0.5)
    logged = [json.loads(l) for l in open(eng.req_log, encoding="utf-8")] if eng.req_log.exists() else []
    meta = next((r for r in logged if r["rid"] == "e11-smoke-a"), {})
    res["a"] = {"tokens": len(got), "finish": fin, "completion_tokens": meta.get("completion_tokens"),
                "text": tokenizer.decode(got)[:80]}
    res["a"]["ok"] = bool(got) and fin in ("stop", "length") and meta.get("completion_tokens") == len(got)
    t1 = time.perf_counter()
    eng.add_request("e11-smoke-b", {"prompt_token_ids": ids}, SamplingParams(max_tokens=4096, temperature=0.6, top_p=0.95, top_k=20))
    got, fin, sent = [], None, False
    while fin is None and time.perf_counter() - t1 < 300:
        for o in eng.step():
            if o.request_id == "e11-smoke-b":
                got += o.outputs[0].token_ids; fin = o.outputs[0].finish_reason if o.finished else None
        if got and not sent:
            eng.abort_request(["e11-smoke-b"]); sent = True; t_abort = time.perf_counter()
        if sent and time.perf_counter() - t_abort > 60:
            break
    res["b"] = {"tokens": len(got), "finish": fin, "abort_sent": sent,
                "abort_to_finish_sec": round(time.perf_counter() - t_abort, 2) if sent else None}
    res["b"]["ok"] = sent and fin == "abort" and len(got) < 4096
    res["ok"] = res["a"]["ok"] and res["b"]["ok"]
    res["sec"] = round(time.perf_counter() - t0, 1)
    return res


def e11_bring_up():
    global E11_SERVER
    E11_SGL["stage"] = "inputs"
    inp = {"int4": SGL_MODEL_PATH, "dflash2": SGL_DRAFT_PATH, "wheelhouse": SGL_WHEELHOUSE}
    E11_SGL["inputs"] = {k: str(v) for k, v in inp.items()}
    missing = [k for k, v in inp.items() if not (v / ("BUILD_INFO.json" if k == "wheelhouse" else "config.json")).is_file()]
    if missing:
        raise E11Fail(f"inputs missing: {missing}")
    if (_load_json(SGL_MODEL_PATH / "config.json").get("quantization_config") or {}).get("quant_method") != "compressed-tensors":
        raise E11Fail("INT4 checkpoint is not compressed-tensors")
    E11_SGL["stage"] = "install"
    vpy, env, E11_SGL["install"] = e11_install()
    launch = [str(vpy), "-m", "sglang.launch_server"] if not E11_HARNESS else [sys.executable, os.environ["E11_FAKE_SERVER"]]
    e14_after_install(vpy, env)             # E14: NCCL bench while the copy runs, then the staged model paths
    E14_VENV["launch"] = list(launch)
    cfg, used = dict(W1), []
    for i in range(3):
        left = SGL_GIVE_UP_AT - time.perf_counter()
        if left < 60:
            raise E11Fail("give-up deadline reached before a launch")
        E11_SGL["stage"] = f"launch {i}"
        srv = E11Server(launch + e11_flags(cfg), env, f"try{i}")
        srv.start()
        # E14-0: the first launch may use the time up to the give-up (the stall rule decides), relaunches keep 20 min
        ok, why = srv.wait_ready(left if i == 0 else min(SGL_RELAUNCH_TIMEOUT_SEC, left))
        att = {"try": i, "cfg": dict(cfg), "ready": ok, "why": why, "sec": round(time.perf_counter() - srv.t0, 1)}
        E11_SGL["attempts"].append(att)
        print(f"E11 SGLang launch {i} {cfg}: {why}", flush=True)
        if ok:
            E11_SERVER = srv
            break
        tail = srv.log_text()[-20000:]
        att["log_tail"] = tail[-3000:]
        srv.kill()
        att["gpu_free"] = _e11_gpu_free()
        nxt = e11_next(cfg, tail, used)
        if nxt is None:
            raise E11Fail(f"launch {i} failed ({why}); no diagnosed fallback")
        cfg, reason = nxt
        used.append(reason); att["retry_reason"] = reason
    else:
        raise E11Fail("3 launches failed")
    E11_SGL["launch_flags"] = e11_flags(cfg)
    E14_VENV["cfg"] = dict(cfg)
    E14_STATE.enter("READY", ready=getattr(E11_SERVER, "e14_stats", None))
    caps = e11_caps(E11_SERVER.log_text())
    caps["gate"], caps["concurrency"] = e11_capacity_gate(caps, cfg)
    E11_SGL["caps"] = caps
    print("E11 caps:", json.dumps(caps), flush=True)
    if not caps["gate"]["pass"]:
        raise E11Fail("capacity gate failed: " + json.dumps(caps["gate"]))
    E11_SGL["stage"] = "smoke"
    eng = E11SglangEngine(E11_SERVER)
    E11_SGL["smoke"] = e11_smoke(eng)
    print("E11 smoke:", json.dumps(E11_SGL["smoke"]), flush=True)
    if not E11_SGL["smoke"]["ok"]:
        raise E11Fail("smoke failed: " + json.dumps(E11_SGL["smoke"])[:600])
    return eng, caps


_t_engine = time.perf_counter()
if not E10_SANDBOX_SELFTEST.get("ok", False):
    E11_SGL["stage"] = "skipped: sandbox self-test failed"
elif ENGINE_BACKEND != "sglang":
    E11_SGL["stage"] = f"skipped: ENGINE_BACKEND={ENGINE_BACKEND}"
else:
    try:
        engine, _caps = e11_bring_up()
        llm = E11SglangLLM()
        ENGINE_READY = True
        CONCURRENCY = int(_caps["concurrency"])                     # rev 2: from the capacity gate (W1: 12)
        # rev 2: SGLang-only clock and KV gate (see CELL 1): drain 12 min before the hard stop; gate from 10 running
        if "drain_margin_sec" in P and P["drain_margin_sec"] < SGL_DRAIN_MARGIN_SEC:
            DRAIN_DEADLINE = HARD_STOP - SGL_DRAIN_MARGIN_SEC
            ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
            assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP
        # rev 3b (user 2026-10-18): request length limit = the server's context (155,648, the E11-04 launch), so the
        # 60k thinking floor never forces a feedback trim or a skip (room for prompts up to ~87k; real max seen 28k).
        # phase1_cap() and phase 2 read MAX_MODEL_LEN at call time; the vLLM fallback keeps S1-60K's 106,496.
        MAX_MODEL_LEN = SGL_CONTEXT_LEN
        PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
        assert phase1_cap(38049) == THINK_BUDGET and phase1_cap(80000) == THINK_BUDGET
        E10_KV_GATE_MIN_ACTIVE = min(E10_KV_GATE_MIN_ACTIVE, SGL_KV_GATE_MIN_ACTIVE)
        E10_KV_ADMIT_MAX = min(E10_KV_ADMIT_MAX, SGL_KV_ADMIT_MAX)
        E11_SGL.update(ok=True, stage="ready")
        ENGINE_INFO.update(backend="sglang-0.5.21-int4-dflash2-w1", kv_tokens=_caps.get("kv_tokens"),
                           effective_running=_caps.get("effective_running"),
                           drain_before_hard_stop_sec=round(HARD_STOP - DRAIN_DEADLINE), max_model_len=MAX_MODEL_LEN,
                           kv_gate={"min_active": E10_KV_GATE_MIN_ACTIVE, "admit_max": E10_KV_ADMIT_MAX})
    except Exception as _e:
        E11_SGL["error"] = f"{type(_e).__name__}: {_e}"[:3000]
        E11_SGL["traceback"] = traceback.format_exc()[-3000:]
        print("E11 SGLang engine NOT ready:", E11_SGL["stage"], "|", E11_SGL["error"][:1500], flush=True)
        e11_kill_server(); E11_SERVER = None
        engine = None; llm = None; ENGINE_READY = False; CONCURRENCY = 0
ENGINE_INFO["load_sec"] = round(time.perf_counter() - _t_engine, 1)
ENGINE_INFO["concurrency"] = int(CONCURRENCY)
print("engine:", json.dumps(ENGINE_INFO, default=str)[:3000], flush=True)


In [ ]:
# ==================================================================== CELL 5b — E14-A1: engine measurements before the Duck run
# Plan: Notes/13, corrected after GPT audits 17-19 (Notes/20). Arms in order, each its own launch: W1 unchanged
# (baseline: windows 8k/40k, profiler, latch test, E14-P token replay, greedy parity serial / A/A / 12 at once), then
# +#44401 (+#41872) backport, +nodelay, + CUDA graphs 1..13. Every arm runs a short correctness gate against the W1
# greedy rows and is adopted only if every recorded condition holds (decisions[*].reason). A1 never eats the reserve of
# the 25-min Duck run; the Duck run (CELL 6) starts on a fresh launch of the kept arm.
import statistics

# ---- E14 shared helpers (CELL 5b and 6b)
def e14_edit_flags(flags, drop=(), set_=None, add=()):
    """drop: flag names removed with their value(s); set_: {flag: value | [values]} replaced or added; add: extra args."""
    out, i, set_ = [], 0, dict(set_ or {})
    names = set(drop) | set(set_)
    while i < len(flags):
        f = flags[i]
        if f in names:
            i += 1
            while i < len(flags) and not str(flags[i]).startswith("--"):
                i += 1
            continue
        out.append(f)
        i += 1
    for k, v in set_.items():
        out += [k] + ([str(x) for x in v] if isinstance(v, (list, tuple)) else ([str(v)] if v is not None else []))
    return out + list(add)


def e14_spec_drop(flags):
    return e14_edit_flags(flags, drop=[f for f in flags if str(f).startswith("--speculative-")])


E14_ARM = {"name": "W1", "set": {}, "drop": (), "env": {}, "spec": True}    # what the server runs now
E14_KEPT = dict(E14_ARM)        # the last ADOPTED arm: next arms build on it; failures and the Duck launch restore it
E14_KEPT_WIN = {}               # the kept arm's windows (short / long / latch): the reference for the next arm
E14_RESERVE = 0.0               # seconds kept for later work (the 25-min Duck run while A1 measures; GPT audits 17-19)
E14_DIRTY = False               # a step left requests running on the server: relaunch before the next phase
E14_GATE_REF = None             # baseline greedy rows for the per-arm correctness gate
E14_LOG_SIGNS = ("runs out of memory", "Not enough space", "Retract", "out of memory", "Traceback", "Exception")


def e14_launch(name, set_=None, drop=(), env=None, spec=True, patches=None, force=False):
    """Kill the current server, launch W1 + this arm's changes, wait (stall watchdog), record caps. A launch that
    could eat the Duck reserve is skipped (force=True: a launch that restores the kept arm or starts the Duck run).
    On a failure the KEPT arm is launched again (never the last launched one, which may have been rejected).
    Returns (ok, record)."""
    global E11_SERVER, E14_ARM, E14_DIRTY
    rec = {"arm": name, "set": set_ or {}, "drop": list(drop), "env": env or {}, "patches": patches, "force": force}
    if not E14M.launch_allowed(time.perf_counter(), HARD_STOP, reserve=0.0 if force else E14_RESERVE, force=force):
        rec["skipped"] = "would eat the Duck reserve" if (not force and E14_RESERVE) else "less than 15 min before the hard stop"
        E14_RESULTS.setdefault("launches", []).append(rec)
        e14_save()
        print(f"E14 launch {name}: skipped ({rec['skipped']})", flush=True)
        return False, rec
    E14_STATE.enter("LAUNCH:" + name, deadline=HARD_STOP)
    t0 = time.perf_counter()
    try:
        e11_kill_server(wait=True)
    except Exception:
        pass
    rec["gpu_free"] = _e11_gpu_free()
    flags = e11_flags(E14_VENV["cfg"])
    flags = e14_edit_flags(flags, drop=drop, set_=set_)
    if not spec:
        flags = e14_spec_drop(flags)
    envx = dict(E14_VENV["env"])
    envx.update(env or {})
    srv = E11Server(E14_VENV["launch"] + flags, envx, "e14_" + name)
    srv.start()
    # a measurement arm may not wait into the Duck reserve (review 3); the Duck / restore launches keep the old bound
    _room = HARD_STOP - time.perf_counter() - (10 * 60 if force else E14_RESERVE + 8 * 60)
    ok, why = srv.wait_ready(min(SGL_RELAUNCH_TIMEOUT_SEC, max(60.0, _room)))
    rec.update(ok=ok, why=why, sec=round(time.perf_counter() - t0, 1), flags=flags, watchdog=getattr(srv, "e14_stats", None))
    if ok:
        E11_SERVER = srv
        caps = e11_caps(srv.log_text())
        rec["caps"] = caps
        if caps.get("kv_tokens"):
            ENGINE_INFO["kv_tokens"] = caps["kv_tokens"]
        E14_ARM = {"name": name, "set": set_ or {}, "drop": tuple(drop), "env": env or {}, "spec": spec}
        E14_DIRTY = False
    else:
        rec["log_tail"] = srv.log_text()[-3000:]
        srv.kill()
        if force and E14_KEPT["name"] != name and not name.startswith("kept_"):
            okb, rb = e14_launch("kept_" + E14_KEPT["name"], set_=E14_KEPT["set"], drop=E14_KEPT["drop"],
                                 env=E14_KEPT["env"], spec=E14_KEPT["spec"], force=True)
            rec["relaunched_kept"] = {"ok": okb, "arm": E14_KEPT["name"], "sec": rb.get("sec")}
    E14_RESULTS.setdefault("launches", []).append(rec)
    e14_save()
    print(f"E14 launch {name}: {why} ({rec['sec']} s)", flush=True)
    return ok, rec


def e14_keep(**windows):
    """Adopt the arm the server runs now: later arms build on it and failures restore it."""
    global E14_KEPT
    E14_KEPT = dict(E14_ARM)
    E14_KEPT_WIN.update({k: v for k, v in windows.items() if v})


def _e14_dirty(obj):
    """True if any cleanup inside a phase result says requests were still running on the server."""
    if isinstance(obj, dict):
        if obj.get("server_idle") is False or obj.get("server_not_idle") is True:
            return True
        return any(_e14_dirty(v) for v in obj.values())
    if isinstance(obj, list):
        return any(_e14_dirty(v) for v in obj[:20])
    return False


def e14_phase(name, fn, cap_sec):
    """Run one phase with its own deadline: min(cap, time left before the hard stop - 8 min - E14_RESERVE); skipped
    (recorded) when less than 2 min would be left. A dirty server from the previous step is relaunched first (no
    carry-over into this phase's windows). Every result and error is recorded and saved at once."""
    global E14_DIRTY
    cap = E14M.phase_budget(time.perf_counter(), HARD_STOP, cap_sec, reserve=E14_RESERVE, margin=8 * 60)
    if cap is not None and E14_DIRTY:       # never measure on a server with leftover requests (review 4: not forced)
        e14_launch(E14_ARM["name"].split("_clean")[0] + "_clean", set_=E14_ARM["set"], drop=E14_ARM["drop"],
                   env=E14_ARM["env"], spec=E14_ARM["spec"])
        cap = E14M.phase_budget(time.perf_counter(), HARD_STOP, cap_sec, reserve=E14_RESERVE, margin=8 * 60)
    if cap is None or E14_DIRTY:
        E14_RESULTS["phases"][name] = {"skipped": "dirty server" if E14_DIRTY else "Duck reserve / hard stop",
                                       "reserve_sec": E14_RESERVE, "left_sec": round(HARD_STOP - time.perf_counter())}
        e14_save()
        print(f"E14 phase {name}: skipped {E14_RESULTS['phases'][name]}", flush=True)
        return None
    E14_CTX.deadline = time.perf_counter() + cap
    E14_STATE.enter("PHASE:" + name, deadline=E14_CTX.deadline, arm=E14_ARM["name"])
    t0 = time.perf_counter()
    _srv = E11_SERVER
    _off = len(_srv.log_text()) if _srv is not None else 0
    try:
        res = fn()
    except BaseException as _e:
        res = {"error": f"{type(_e).__name__}: {_e}"[:500], "traceback": traceback.format_exc()[-1500:]}
    if isinstance(res, dict):
        res.setdefault("arm", E14_ARM["name"])
        res["phase_sec"] = round(time.perf_counter() - t0, 1)
        res["budget"] = {"cap_sec": round(cap, 1), "reserve_sec": E14_RESERVE,
                         "end_before_hard_stop_sec": round(HARD_STOP - E14_CTX.deadline, 1)}
        if _srv is not None:                 # over-admission / eviction signs (substring diagnostic, not a classifier)
            _new = _srv.log_text()[_off:]
            res["server_log_signs"] = {s: _new.count(s) for s in E14_LOG_SIGNS if s in _new}
        if _e14_dirty(res):
            E14_DIRTY = True
            res["server_not_idle_after"] = True
    E14_RESULTS["phases"][name] = res
    e14_save()
    print(f"E14 phase {name}: {json.dumps(res, default=str)[:900]}", flush=True)
    return res


def e14_decide(name, conds, **extra):
    """conds: [(label, bool)]: adopted only if all hold; the first failing label is the recorded reason."""
    why = next((lab for lab, ok in conds if not ok), None)
    E14_RESULTS["decisions"][name] = dict(adopt=why is None, reason="all conditions met" if why is None else why,
                                          conditions={lab: bool(ok) for lab, ok in conds}, **extra)
    return why is None


def _e14_model_id():
    try:
        return json.loads(_urlreq.urlopen(SGL_URL + "/v1/models", timeout=30).read())["data"][0]["id"]
    except Exception:
        return str(SGL_MODEL_PATH)


def _e14_rows(g):
    return "\n".join(" ".join(str(v) for v in r) for r in g)


def _e14_task_text(t):
    s = "".join(f"\nExample {i + 1}\nInput:\n{_e14_rows(p['input'])}\nOutput:\n{_e14_rows(p['output'])}\n"
                for i, p in enumerate(t["train"]))
    return s + "".join(f"\nTest input {j + 1}:\n{_e14_rows(q['input'])}\n" for j, q in enumerate(t["test"]))


def _e14_chat_ids(messages, tools=None, think=True):
    txt = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                        enable_thinking=think, reasoning_effort=REASONING_EFFORT)
    return tokenizer.encode(txt, add_special_tokens=False)


def e14_prompts():
    """short: one real task per stream (the ARC text the model reasons about); long: a shared ~36k-token block of other
    eval tasks + the task (the 40k band); parity: the 12 task prompts, thinking on; Duck: system prompt + tools + first
    opener exactly as the solver sends them."""
    ask = "Find the transformation rule of this ARC-AGI-2 task and describe it, then give the test output grids."
    short = [_e14_chat_ids([{"role": "user", "content": ask + "\n" + _e14_task_text(CHALLENGES[t])}]) for t in TASK_IDS]
    try:
        _all = _load_json(ARC_ROOT / "arc-agi_evaluation_challenges.json")
    except Exception:
        _all = dict(CHALLENGES)
    bg, n = [], 0
    for t in sorted(_all):
        if t in CHALLENGES:
            continue
        txt = _e14_task_text(_all[t])
        bg.append(txt)
        n += len(tokenizer.encode(txt, add_special_tokens=False))
        if n >= int(os.environ.get("E14_LONG_TOKENS", 36000)):
            break
    block = "Reference tasks (not to be solved):\n" + "\n".join(bg)
    long_ = [_e14_chat_ids([{"role": "system", "content": block},
                            {"role": "user", "content": ask + "\n" + _e14_task_text(CHALLENGES[t])}]) for t in TASK_IDS]
    TA, _, _, wrs = E12A._import_duck()
    Agent = E12A.make_agent_class()
    duck = []
    for t in TASK_IDS:
        ag = Agent(task=CHALLENGES[t], task_id=t, branch=0, view="facts", model=E14_CTX.model_id, base_url=SGL_URL + "/v1",
                   provider="vllm")
        d = E14_OUT / f"duck_probe_{t}"
        d.mkdir(parents=True, exist_ok=True)
        sp = d / TA.RUNTIME_STATE_FILENAME
        wrs(sp, current_frame=None, history=[])
        duck.append((ag._system_prompt, E12A.R6_TEXT["first"] + E12A.R6_COMPLETE, ag._tools(sp)))
    tool_text = "Tool output:\n" + _e14_rows(CHALLENGES[TASK_IDS[0]]["train"][0]["input"])
    return {"short": short, "long": long_, "duck": duck, "tool_text": tool_text,
            "tokens": {"short_p50": statistics.median(len(x) for x in short) if short else None,
                       "long_p50": statistics.median(len(x) for x in long_) if long_ else None}}

E14_CTX = None
if E14_DO_MEASURE and ENGINE_READY:
    E14_CTX = E14M.Ctx(SGL_URL, lambda: E11_SERVER.log_text() if E11_SERVER is not None else "", E14_OUT,
                       model_id=_e14_model_id(), deadline=HARD_STOP - 8 * 60, log=print, encode=None,
                       normalize=_E12_TA._normalize_message_content)
    E14_CTX.scale = E14_TS
    E14_KEPT = dict(E14_ARM)
    # Duck reserve (GPT audits 17 HIGH-2, 18 P0, 19 HIGH-E): A1 phases and A1 relaunches never eat the 25-min Duck run:
    # its window + the solver stop margin + preflight + final artifacts + one conservative (cold capture) relaunch
    _rs = {"solve_window": float(E12_SOLVE_WINDOW_SEC or 1500), "stop_before_hard": float(E12_STOP_BEFORE_HARD_SEC),
           "preflight": _ts(300), "artifacts": _ts(600), "duck_relaunch": _ts(400)}
    E14_RESERVE = round(sum(_rs.values()), 1)
    E14_RESULTS["a1_reserve"] = dict(_rs, total=E14_RESERVE)
    _t = time.perf_counter()
    try:
        E14_PROMPTS = e14_prompts()
    except Exception as _e:           # review 13: never let a prompt-builder error stop the notebook before the Duck run
        E14_PROMPTS = {"short": [], "long": [], "duck": [], "tool_text": "", "tokens": {"error": repr(_e)[:300]}}
    E14_RESULTS["prompts"] = dict(E14_PROMPTS["tokens"], sec=round(time.perf_counter() - _t, 1))
    _S, _L = E14_PROMPTS["short"], E14_PROMPTS["long"]
    _summ = lambda d, s: E14T.summarize(d, s)
    _latch = lambda: E14M.latch_test(E14_CTX, _S, _L, decodes=SGL_MAX_RUNNING - 1, arrivals=3, secs=_ts(60),
                                     server_cap=SGL_SERVER_MAX_RUNNING, level=SGL_MAX_RUNNING)

    def _a1_baseline():
        out = {"short": E14M.steady_window(E14_CTX, _S, level=SGL_MAX_RUNNING, secs=_ts(75), tag="w1short"),
               "long": E14M.steady_window(E14_CTX, _L, level=SGL_MAX_RUNNING, secs=_ts(75), tag="w1long")}
        out["profile"] = E14M.steady_window(E14_CTX, _S, level=SGL_MAX_RUNNING, secs=_ts(60), tag="w1prof",
                                            during=lambda c: E14M.profile_steps(c, steps=20, summarize=_summ))
        out["latch"] = _latch()
        return out
    E14_BASE = e14_phase("A1-2_baseline_W1", _a1_baseline, _ts(15 * 60))
    e14_keep(**{k: (E14_BASE or {}).get(k) for k in ("short", "long", "latch")})
    e14_phase("A1-3_replay_E14P", lambda: E14M.replay_check(E14_CTX, E14_PROMPTS["duck"][:3]), _ts(8 * 60))

    def _a1_parity():
        a = E14M.parity_pass(E14_CTX, _S, max_new=512, tag="d_serial_a")
        b = E14M.parity_pass(E14_CTX, _S, max_new=512, tag="d_serial_b")
        c = E14M.parity_pass(E14_CTX, _S, max_new=512, concurrent=True, tag="d_c12")
        c2 = E14M.parity_pass(E14_CTX, _S, max_new=512, concurrent=True, tag="d_c12_b")
        _dirty = any(x and x[0].get("server_not_idle") for x in (a, b, c, c2))
        # every later arm gate compares with these; a reference taken beside leftover requests is no reference (review 12)
        globals()["E14_GATE_REF"] = None if _dirty else E14M.gate_reference(a, b, c, c2)
        # batching alone changes greedy tokens (E12: 152-166 of 512 shared, no fault): concurrency 12 gets its own A/A;
        # the decision rests on large-margin flips (> 1 nat) judged against the A/A, never on raw token equality
        return {"aa": E14M.compare_passes(a, b, "D serial A vs D serial B (A/A)"),
                "aa_c12": E14M.compare_passes(c, c2, "D c12 vs D c12 (A/A under batching)"),
                "contamination": E14M.compare_passes(a, c, "D serial vs D at concurrency 12"),
                "server_not_idle": _dirty, "gate_reference": "invalid (dirty server)" if _dirty else "ok"}
    E14_PAR = e14_phase("A1-4_parity_D", _a1_parity, _ts(12 * 60))
    if isinstance(E14_PAR, dict) and "aa" in E14_PAR:
        E14_RESULTS["decisions"]["dflash_contamination"] = E14M.parity_ok(E14_PAR["contamination"], E14_PAR["aa_c12"],
                                                                          allowance=2, identity="ignore")

    def _arm_windows(tag, latch=True, extra=None):
        """The correctness gate FIRST (audits 17-19: correctness before any throughput arm; a gate placed after the
        windows was once starved of time and rejected a good arm as INCOMPLETE), then the windows only if it passed."""
        def run():
            # identity=False: DFlash2 greedy output is byte-identical within one process but can differ across a
            # restart (#38009, Notes/Research - 4xL4 ...); every arm is a new process, so large-margin flips vs the
            # A/A decide (a corrupt arm still fails: 2.4-nat flips) and identity is only recorded (review 1)
            out = {"gate": E14M.arm_gate(E14_CTX, _S, E14_GATE_REF, tag="gate_" + tag, identity=False)}
            if not out["gate"].get("ok"):
                out["windows_skipped"] = "the correctness gate did not pass"
                return out
            out["flush_after_gate"] = E14M.flush_cache(E14_CTX)   # the windows start uncached like W1's (review 14)
            out["short"] = E14M.steady_window(E14_CTX, _S, level=SGL_MAX_RUNNING, secs=_ts(75), tag=tag + "short")
            out["long"] = E14M.steady_window(E14_CTX, _L, level=SGL_MAX_RUNNING, secs=_ts(75), tag=tag + "long")
            if latch:
                out["latch"] = _latch()
            out.update((extra or (lambda: {}))())
            return out
        return run

    # ---- arm 1: #44401 + #41872 backport (source-hash guarded; refused = arm skipped). Adopted only with a real
    # occupancy gain at the same offered load, a passing correctness gate and no server error signs.
    _site = None
    if not E11_HARNESS:
        try:
            _site = subprocess.run([E14_VENV["vpy"], "-c", "import sglang, os; print(os.path.dirname(sglang.__file__))"],
                                   capture_output=True, text=True, timeout=120).stdout.strip()
        except Exception as _e:
            _site, E14_RESULTS["patch_site_error"] = None, repr(_e)[:200]
    else:
        _site = os.environ.get("E14_FAKE_SGLANG_PKG")
    _pr = E14P.apply(_site, names=("44401", "41872")) if _site else {"error": "sglang package path unknown"}
    E14_RESULTS["patches"] = _pr
    _applied = [n for n, v in (_pr.get("results") or {}).items() if v.get("status") in ("applied", "already applied")]
    if "44401" in _applied:
        _ok, _ = e14_launch("p44401", set_=E14_KEPT["set"], drop=E14_KEPT["drop"], env=E14_KEPT["env"], patches=_applied)
        _pp = e14_phase("A1-5_p44401", _arm_windows("p44401"), _ts(14 * 60)) if _ok else None
        _sp = E14M.adopt_speed(E14_KEPT_WIN.get("short"), (_pp or {}).get("short"), occupancy=True, per_step=True)
        _lg = E14M.latch_gain(E14_KEPT_WIN.get("latch"), (_pp or {}).get("latch"))
        _gt = (_pp or {}).get("gate") or {"ok": False, "why": "not run"}
        _er = E14M.log_errors((_pp or {}).get("server_log_signs"))
        if e14_decide("p44401", [("launched and measured", bool(_ok and _pp and not _pp.get("error"))),
                                 ("correctness gate", _gt.get("ok")),
                                 ("occupancy gain", _lg["gain"]), ("speed not worse", _sp["adopt"]),
                                 ("no server error signs", not _er)],
                      speed=_sp, latch_gain=_lg, gate=_gt, log_errors=_er):
            e14_keep(**{k: _pp.get(k) for k in ("short", "long", "latch")})
    else:
        e14_decide("p44401", [("patch applied", False)], report=_pr)
    if _applied and not (E14_RESULTS["decisions"].get("p44401") or {}).get("adopt"):
        # every applied hunk that was not adopted is reverted (#41872 alone included; review 7); the revert is checked
        _rv = E14P.revert(_site)
        E14_RESULTS["patches_reverted"] = _rv
        if not (isinstance(_rv, dict) and _rv.get("ok")):
            E14_RESULTS["patch_state"] = "revert NOT confirmed: later launches may still run patched files"

    # ---- arm 1b: the DFlash min-free-slots delayer off (Notes/14; source: managers/min_free_slots_delayer.py +
    # scheduler.py L1182-1193). Unset + DFlash -> threshold min(4, max(2, (13 + 5) // 6)) = 3: with 11 running only 2
    # slots are allocatable, so a returning stream waits until a second one finishes = the 11 running + 1 queued state.
    # --min-free-slots-delay 1 disables it ("<= 1 disables"). Needs a real 11 -> 12 gain (latch_gain), not a tie.
    _ok, _ = e14_launch("nodelay", set_=dict(E14_KEPT["set"], **{"--min-free-slots-delay": 1}), drop=E14_KEPT["drop"],
                        env=E14_KEPT["env"])
    _np = e14_phase("A1-5b_nodelay", _arm_windows("nodelay"), _ts(14 * 60)) if _ok else None
    _sp = E14M.adopt_speed(E14_KEPT_WIN.get("short"), (_np or {}).get("short"), occupancy=True, per_step=True)
    _lg = E14M.latch_gain(E14_KEPT_WIN.get("latch"), (_np or {}).get("latch"))
    _gt = (_np or {}).get("gate") or {"ok": False, "why": "not run"}
    _er = E14M.log_errors((_np or {}).get("server_log_signs"))
    if e14_decide("nodelay", [("launched and measured", bool(_ok and _np and not _np.get("error"))),
                              ("correctness gate", _gt.get("ok")),
                              ("occupancy gain", _lg["gain"]), ("speed not worse", _sp["adopt"]),
                              ("no server error signs", not _er)],
                  speed=_sp, latch_gain=_lg, gate=_gt, log_errors=_er):
        e14_keep(**{k: _np.get(k) for k in ("short", "long", "latch")})

    # ---- arm 2: CUDA graphs for every batch size 1..13 (13 = the server's admission limit), on top of the kept arms;
    # adopted only if the server really captured 12 and 13 (fail closed on an unparsed list)
    _gbs = {"--cuda-graph-bs-decode": list(range(1, SGL_SERVER_MAX_RUNNING + 1))}
    _ok, _grec = e14_launch("graphs13", set_=dict(E14_KEPT["set"], **_gbs),
                            drop=tuple(E14_KEPT["drop"]) + ("--cuda-graph-max-bs-decode",), env=E14_KEPT["env"])
    _gcap = ((_grec or {}).get("caps") or {}).get("verify_graph_bs")      # what the server really captured
    _gp = e14_phase("A1-6_graphs13", _arm_windows("g13", latch=False, extra=lambda: {
        "short13": E14M.steady_window(E14_CTX, _S, level=SGL_SERVER_MAX_RUNNING, secs=_ts(60), tag="g13at13")}),
        _ts(12 * 60)) if _ok else None
    _gs = E14M.adopt_speed(E14_KEPT_WIN.get("short"), (_gp or {}).get("short"), per_step=True)
    _gcov = bool(_gcap) and {12, SGL_SERVER_MAX_RUNNING} <= set(_gcap)
    _gt = (_gp or {}).get("gate") or {"ok": False, "why": "not run"}
    _er = E14M.log_errors((_gp or {}).get("server_log_signs"))
    if e14_decide("graphs13", [("launched and measured", bool(_ok and _gp and not _gp.get("error"))),
                               ("correctness gate", _gt.get("ok")),
                               ("captured 12 and 13", _gcov), ("speed not worse", _gs["adopt"]),
                               ("no server error signs", not _er)],
                  speed=_gs, gate=_gt, log_errors=_er, captured_bs=_gcap):
        e14_keep(**{k: _gp.get(k) for k in ("short", "long")})

    # ---- the Duck run starts on a FRESH launch of the kept arm: an empty radix cache and Mamba state like the hidden
    # run, and the kept arm restored after a rejected one (audit 17 HIGH-4 / 18 Q3 / 19 §3). /flush_cache only when the
    # relaunch did not happen; its plain-text reply is classified, never assumed.
    E14_RESERVE = 0.0
    _okf, _ = e14_launch("duck_" + E14_KEPT["name"], set_=E14_KEPT["set"], drop=E14_KEPT["drop"], env=E14_KEPT["env"],
                         spec=E14_KEPT["spec"], force=True)
    _fresh = bool(_okf) or (E14_ARM["name"].startswith("kept_") and bool((E14_RESULTS.get("launches") or [{}])[-1].get("ok")))
    _clean = {"method": "relaunch" if _okf else ("relaunch (kept arm after a failed Duck launch)" if _fresh else "relaunch failed"),
              "ok": _fresh, "arm": E14_ARM["name"]}
    if not _fresh:
        E14_CTX.deadline = time.perf_counter() + max(5.0, min(120.0, HARD_STOP - time.perf_counter() - 8 * 60))
        _idle = E14M.server_idle(E14_CTX, wait=30.0)
        try:
            _msg = (E14M.post_json(E14_CTX, "/flush_cache", {}, timeout=60) or {}).get("text") or ""   # http_server.py:987
        except Exception as _e:
            _msg = "error: " + repr(_e)[:200]
        _fl = _msg.strip().startswith("Cache flushed")
        _clean = {"method": "flush", "idle_before": _idle, "ok": _fl, "reply": _msg.strip()[:300], "arm": E14_ARM["name"]}
    _clean["cold_start_unverified"] = not _clean["ok"]
    _clean["kept_arm_running"] = E14_ARM["set"] == E14_KEPT["set"] and E14_ARM["drop"] == E14_KEPT["drop"] and \
        E14_ARM["env"] == E14_KEPT["env"]
    E14_RESULTS["duck_clean_start"] = _clean
    _clean["server_reachable_and_idle"] = E14M.server_idle(E14_CTX, wait=5.0)
    _avail = HARD_STOP - E12_STOP_BEFORE_HARD_SEC - time.perf_counter()
    E14_RESULTS["duck_window"] = {"planned_sec": E12_SOLVE_WINDOW_SEC, "available_sec": round(_avail),
                                  "status": "full" if _avail >= (E12_SOLVE_WINDOW_SEC or 0) + _ts(300)
                                  else "solver_window_unavailable"}
    E14_RESULTS["duck_arm"] = dict(E14_ARM)
    E14_STATE.enter("DUCK", arm=E14_ARM["name"])
    e14_save()
    print("E14-A1 decisions:", json.dumps({k: (v.get("adopt", v.get("ok")), v.get("reason", v.get("verdict")))
                                           for k, v in E14_RESULTS["decisions"].items()
                                           if isinstance(v, dict)}, default=str)[:1500],
          "| Duck runs on", E14_ARM["name"], "| clean start", _clean, "| window", E14_RESULTS["duck_window"], flush=True)
elif E14_DO_MEASURE:
    E14_RESULTS["phases"]["A1"] = {"skipped": f"engine not ready (ENGINE_READY={ENGINE_READY})"}
    e14_save()


In [ ]:
# ==================================================================== CELL 6 — E12: Duck REPL agent, every task, E12_BRANCHES branches
# Every branch is admitted at once; Duck's priority gate keeps ARC3_MAX_ACTIVE_STREAMS conversations active (first
# branches in ORDER2 order before any second branch). A branch ends when the host verifies a submitted program on every
# train pair, at the deadline, or at the optional token cap. Full thinking: Duck's transcript.txt per branch.
_E12_INDEX = {t: i for i, t in enumerate(TASK_IDS)}


def _e12_order2_score(tid):   # = S49 ORDER2 ([>=3 train] + [1 test]) with the cells tie-break, sorted-id order last
    _t = CHALLENGES[tid]
    return int(len(_t["train"]) >= 3) + int(len(_t["test"]) == 1)


def _e12_total_cells(tid):
    _t = CHALLENGES[tid]
    _c = lambda g: len(g) * len(g[0]) if g and g[0] else 0
    return sum(_c(p["input"]) + _c(p["output"]) for p in _t["train"]) + sum(_c(p["input"]) for p in _t["test"])


E12_ORDER = sorted(TASK_IDS, key=lambda t: (-_e12_order2_score(t), _e12_total_cells(t), _E12_INDEX[t]))
E12_RECORDS = {t: [] for t in TASK_IDS}
E12_RUN = {"started_sec": None, "model_id": None, "streams": None, "deadline_in_sec": None, "error": None,
           "branches_done": 0, "branches_accepted": 0, "finished_sec": None}
_E12_LOCK = threading.Lock()
E12_OUT = OUT / "duck"
E12_OUT.mkdir(parents=True, exist_ok=True)


import tempfile as _tempfile
_E12_WRITE_LOCK = threading.Lock()   # rev 2 (GPT audits, reproduced): branches finish concurrently


def _e12_write_json_atomic(path, obj):
    """A unique temp file per write + os.replace (CELL 0's helper reuses one '<name>.tmp' for every writer)."""
    fd, tmp = _tempfile.mkstemp(prefix=Path(path).name + ".", suffix=".tmp", dir=str(Path(path).parent))
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(obj, f)
        os.replace(tmp, path)
    except BaseException:
        try:
            os.unlink(tmp)
        except OSError:
            pass
        raise


def e12_write_submission():
    with _E12_WRITE_LOCK:                          # one writer at a time; the snapshot is taken inside, so it is fresh
        sub = identity_submission(test_challenges)
        if RERUN:
            with _E12_LOCK:
                recs = {t: list(v) for t, v in E12_RECORDS.items() if v}
            for t, rows in E12A.build_submission({t: CHALLENGES[t] for t in recs}, recs).items():
                if t in sub and len(sub[t]) == len(rows):
                    sub[t] = rows
        _e12_write_json_atomic(SUBMISSION_PATH, sub)   # evaluation: keeps the competition-format test fallback


def _e12_branch_done_r6(rec):   # rev 6: a branch reports every pause and its finish; keep the latest per branch
    with _E12_LOCK:
        rows = E12_RECORDS.setdefault(rec["task_id"], [])
        # rev 6g H7-1 (GPT rev 6f audits): callbacks can finish out of order; never let an older record replace a newer one
        _old = next((r for r in rows if r.get("branch") == rec.get("branch")), None)
        if _old is not None and int(rec.get("publish_seq") or 0) <= int(_old.get("publish_seq") or 0):
            E12_RUN["stale_callbacks_dropped"] = E12_RUN.get("stale_callbacks_dropped", 0) + 1
            return
        new = rec.get("branch") not in {r.get("branch") for r in rows}
        rows[:] = [r for r in rows if r.get("branch") != rec.get("branch")] + [rec]
        E12_RUN["branches_started"] = E12_RUN.get("branches_started", 0) + int(new)
        E12_RUN["branches_done"] = sum(1 for v in E12_RECORDS.values() for r in v if r.get("status") == "done")
        E12_RUN["branches_accepted"] = sum(1 for v in E12_RECORDS.values() for r in v if r.get("accepted"))
    try:
        e12_write_submission()
    except Exception as _e:
        print("E12 submission write failed:", repr(_e)[:200], flush=True)


def _e12_g4b_window(hidden_solve_end_sec, solve_start_sec, n_tasks, hidden_tasks=240):
    # gate 4b (user 2026-10-21): this run gets the hidden run's compute per task. H6-16: a function so the real
    # values (42,720 / 1,463 / 12 -> 2,063 s) are unit-tested, not only under the harness's 300 s cap
    per = max(0.0, float(hidden_solve_end_sec) - float(solve_start_sec)) / float(hidden_tasks)
    return per, per * int(n_tasks)


def _e12_shifted_end(end, setup_sec, hard_end):
    # SA-1 / H6-6: a commit window starts after the preflight (and nothing else runs before solving since H6-3); the end
    # moves by the setup time, never past the hard stop
    return min(float(hard_end), float(end) + max(0.0, float(setup_sec)))


def _e12_capacity_probe():   # rev 6e G4-1: exact names; the max over TP ranks; never sglang:token_usage (max of pools)
    try:
        rows = [r for r in e11_prom() if getattr(r, "value", None) is not None]
    except Exception:
        rows = []
    pick = lambda name: [float(r.value) for r in rows if r.name == name]
    out = {}
    for key, name in (("kv_usage", "sglang:full_token_usage"), ("mamba_usage", "sglang:mamba_usage"),
                      ("running", "sglang:num_running_reqs"), ("queue", "sglang:num_queue_reqs")):
        v = pick(name)
        out[key] = max(v) if v else None
    return out


def _e12_cache_diag(budget_sec=240.0):
    # rev 6e G4-8 (GPT G4B audits): a bounded exact-prompt replay for the prefix-cache misses (G4B: 20/180 requests with
    # 0 cached tokens while KV <= 0.44; suspects SGLang #22935 split 0-hit and #36935 mamba checkpoint eviction).
    # Prompt A (about 9k tokens) warm -> exact replay -> two divergent suffixes -> 12 distinct long prompts at once
    # (churn) -> replay and a divergent suffix again. max_tokens 1, thinking off. Commit mode only.
    import random as _rnd, concurrent.futures as _cf
    t_end = time.monotonic() + float(budget_sec)

    def _blob(seed, rows=150):
        r = _rnd.Random(seed)
        return "\n".join(" ".join(str(r.randrange(10)) for _ in range(30)) for _ in range(rows))

    def _ask(msgs):
        body = {"model": E12_RUN["model_id"], "max_tokens": 1, "temperature": 0.0, "messages": msgs,
                "chat_template_kwargs": {"enable_thinking": False}}
        t0 = time.monotonic()
        req = _urlreq.Request(SGL_URL + "/v1/chat/completions", data=json.dumps(body).encode("utf-8"),
                              headers={"Content-Type": "application/json"})
        r = json.loads(_urlreq.urlopen(req, timeout=max(5.0, t_end - time.monotonic())).read())
        u = r.get("usage") or {}
        det = u.get("prompt_tokens_details") if isinstance(u.get("prompt_tokens_details"), dict) else {}
        return {"prompt": u.get("prompt_tokens"), "cached": int(det.get("cached_tokens") or 0),
                "cached_reported": "cached_tokens" in det, "sec": round(time.monotonic() - t0, 2)}

    A = {"role": "system", "content": "E12-CACHE-DIAG reference A\n" + _blob(1)}
    q = lambda text: {"role": "user", "content": text}
    steps = []

    def step(name, msgs):
        if time.monotonic() > t_end - 5:
            steps.append({"step": name, "skipped": "budget"})
            return
        try:
            res = _ask(msgs)
        except Exception as _e:
            res = {"error": repr(_e)[:200]}
        res["step"] = name
        res.update({"g_" + k: v for k, v in _e12_capacity_probe().items()})
        steps.append(res)
    step("warm", [A, q("Reply with one word.")])
    step("replay_exact", [A, q("Reply with one word.")])
    step("divergent_1", [A, q("Divergent continuation one. Reply with one word.")])
    step("divergent_2", [A, q("Divergent continuation two. Reply with one word.")])
    if time.monotonic() < t_end - 30:
        churn = [[{"role": "system", "content": f"E12-CACHE-DIAG churn {i}\n" + _blob(100 + i)}, q("Reply with one word.")]
                 for i in range(max(1, int(E12_RUN.get("streams") or 12)))]

        def _safe(m):
            try:
                return _ask(m)
            except Exception as _e:
                return {"error": repr(_e)[:120]}
        with _cf.ThreadPoolExecutor(len(churn)) as _ex:
            cr = list(_ex.map(_safe, churn))
        steps.append({"step": "churn", "n": len(cr), "errors": sum(1 for x in cr if "error" in x),
                      "prompt_sum": sum(int(x.get("prompt") or 0) for x in cr),
                      "sec_max": max((x.get("sec") or 0) for x in cr),
                      **{"g_" + k: v for k, v in _e12_capacity_probe().items()}})
    step("replay_after_churn", [A, q("Reply with one word.")])
    step("divergent_after_churn", [A, q("Divergent continuation three. Reply with one word.")])
    by = {x["step"]: x for x in steps if "prompt" in x}
    pa = (by.get("warm") or {}).get("prompt")
    out = {"budget_sec": budget_sec, "prompt_tokens_A": pa, "steps": steps,
           "summary": {k: (by[k]["cached"], by[k]["prompt"]) for k in ("warm", "replay_exact", "divergent_1", "divergent_2",
                                                                         "replay_after_churn", "divergent_after_churn")
                       if k in by}}
    print("E12 cache diag (cached, prompt):", json.dumps(out["summary"]), flush=True)
    return out


def _e12_no_spec(cmd):   # the same server command without --speculative-* (each takes one value)
    out, skip = [], False
    for x in cmd:
        if skip:
            skip = False
            continue
        if str(x).startswith("--speculative-"):
            skip = True
            continue
        out.append(x)
    return out


def _e12_parity():
    # R6-X1 target-only parity (user 2026-10-21): the rev 6 system prompt + first opener of each task, with tools, T 0,
    # one request at a time, max_tokens 2,048, thinking on and off. DFlash2 twice (control), then a relaunch without
    # speculative decoding. Judged on equality and on what Duck acts on (tool-call presence and arguments, finish).
    global E11_SERVER
    TA, _, _, wrs = E12A._import_duck()
    Agent = E12A.make_agent_class()
    probes = []
    for tid in TASK_IDS:
        ag = Agent(task=CHALLENGES[tid], task_id=tid, branch=0, view="facts", model=E12_RUN["model_id"],
                   base_url=SGL_URL + "/v1", provider="vllm")
        d = E12_OUT / f"parity_{tid}"
        d.mkdir(parents=True, exist_ok=True)
        sp = d / TA.RUNTIME_STATE_FILENAME
        wrs(sp, current_frame=None, history=[])
        for think in (True, False):
            probes.append((tid, think, {"model": E12_RUN["model_id"], "max_tokens": 2048, "temperature": 0.0, "top_p": 1.0,
                                        "messages": [{"role": "system", "content": ag._system_prompt},
                                                     {"role": "user", "content": E12A.R6_TEXT["first"] + E12A.R6_COMPLETE}],
                                        "tools": ag._tools(sp), "tool_choice": "auto",
                                        "chat_template_kwargs": {"preserve_thinking": True, "enable_thinking": think}}))

    def _ask(body):
        req = _urlreq.Request(SGL_URL + "/v1/chat/completions", data=json.dumps(body).encode("utf-8"),
                              headers={"Content-Type": "application/json"})
        r = json.loads(_urlreq.urlopen(req, timeout=900).read())
        m = r["choices"][0]["message"]
        return {"reasoning": str(m.get("reasoning_content") or ""), "content": str(m.get("content") or ""),
                "calls": [str((tc.get("function") or {}).get("arguments", "")) for tc in (m.get("tool_calls") or [])],
                "finish": r["choices"][0].get("finish_reason"), "tokens": (r.get("usage") or {}).get("completion_tokens")}

    def _round(tag):
        rows = []
        for tid, think, body in probes:
            try:
                rows.append(_ask(body))
            except Exception as _e:
                rows.append({"error": repr(_e)[:300]})
        (E12_OUT / f"parity_{tag}.json").write_text(json.dumps(rows), encoding="utf-8")
        return rows
    out = {"probes": [{"task": t, "thinking": k} for t, k, _ in probes]}
    a, b = _round("dflash_a"), _round("dflash_b")
    try:
        cmd, env = list(E11_SERVER.cmd), E11_SERVER.env
        e11_kill_server(wait=True)
        srv = E11Server(_e12_no_spec(cmd), env, "target_only")
        srv.start()
        ok, why = srv.wait_ready(min(SGL_RELAUNCH_TIMEOUT_SEC, max(60, HARD_STOP - time.perf_counter() - 600)))
        E11_SERVER = srv
        out["target_only_launch"] = {"ok": ok, "why": why, "spec_flags_removed": [x for x in cmd if str(x).startswith("--speculative-")]}
        t = _round("target_only") if ok else [{"error": "target-only launch failed"} for _ in probes]
    except Exception as _e:
        out["target_only_launch"] = {"ok": False, "why": repr(_e)[:300]}
        t = [{"error": "target-only launch failed"} for _ in probes]

    def _pre(x, y):
        n = 0
        for c1, c2 in zip(x, y):
            if c1 != c2:
                break
            n += 1
        return n
    cmp_ = []
    for (tid, think, _), ra, rb, rt in zip(probes, a, b, t):
        row = {"task": tid, "thinking": think}
        if "error" not in ra and "error" not in rb:
            row["control_identical"] = ra == rb
        if "error" not in ra and "error" not in rt:
            ta, tt = ra["reasoning"] + ra["content"], rt["reasoning"] + rt["content"]
            row.update(target_identical=ra == rt, common_prefix_chars=_pre(ta, tt), dflash_chars=len(ta), target_chars=len(tt),
                       same_tool_call_presence=bool(ra["calls"]) == bool(rt["calls"]), same_tool_args=ra["calls"] == rt["calls"],
                       same_finish=ra["finish"] == rt["finish"], dflash_tokens=ra["tokens"], target_tokens=rt["tokens"])
        cmp_.append(row)
    out["compare"] = cmp_
    ok_rows = [r for r in cmp_ if "target_identical" in r]
    out["summary"] = {"probes": len(cmp_), "control_identical": sum(1 for r in cmp_ if r.get("control_identical")),
                      "target_identical": sum(1 for r in ok_rows if r["target_identical"]),
                      "same_tool_call_presence": sum(1 for r in ok_rows if r["same_tool_call_presence"]),
                      "same_finish": sum(1 for r in ok_rows if r["same_finish"]), "compared": len(ok_rows)}
    print("E12 parity:", json.dumps(out["summary"]), flush=True)
    return out


def _e12_r6_preflight(n):
    # R6-X1 (bounded, inside this session): the drafter's tap-layer config (SGLang #37475) and an isolation smoke for
    # #36548 (n concurrent requests with distinct markers; a marker in another request's reply = contamination).
    out = {}
    try:
        _cfg = json.loads((SGL_DRAFT_PATH / "config.json").read_text(encoding="utf-8"))
        _flat = dict(_cfg)
        for _v in list(_cfg.values()):
            if isinstance(_v, dict):
                _flat.update(_v)
        out["drafter_target_layer_config"] = {k: v for k, v in _flat.items() if "layer" in k.lower()
                                              and ("target" in k.lower() or "tap" in k.lower())}
    except Exception as _e:
        out["drafter_config_error"] = repr(_e)[:200]
    try:
        import concurrent.futures as _cf, random as _rnd
        marks = ["ZQ" + str(_rnd.randrange(10 ** 6, 10 ** 7)) + "K" + str(i) for i in range(n)]

        def _one(m):
            body = {"model": E12_RUN["model_id"], "max_tokens": 24, "temperature": 0.0,
                    "messages": [{"role": "user", "content": "Reply with exactly this code and nothing else: " + m}],
                    "chat_template_kwargs": {"enable_thinking": False}}
            req = _urlreq.Request(SGL_URL + "/v1/chat/completions", data=json.dumps(body).encode("utf-8"),
                                  headers={"Content-Type": "application/json"})
            r = json.loads(_urlreq.urlopen(req, timeout=180).read())
            return str(r["choices"][0]["message"].get("content") or "")
        with _cf.ThreadPoolExecutor(n) as _ex:
            replies = list(_ex.map(_one, marks))
        out.update(E12A.isolation_verdict(marks, replies))     # C6-2: PASS / FAIL / INCONCLUSIVE
    except Exception as _e:
        out.update(isolation_error=repr(_e)[:300], isolation_state="INCONCLUSIVE", isolation_ok=None)
    try:   # R6-X1 parity CONTROL: the same request twice at T 0, one at a time; equal replies = a deterministic baseline.
        # A target-only (no DFlash) comparison needs a separate server launch and is never run beside production.
        pc = []
        for _m in ("PARITY-A-" + str(n), "PARITY-B-" + str(n)):
            _a, _b = _one(_m), _one(_m)
            pc.append({"same": _a == _b, "own_marker": _m in _a})
        out["parity_control"] = {"pairs": len(pc), "identical": sum(x["same"] for x in pc),
                                 "own_marker": sum(x["own_marker"] for x in pc),
                                 "note": "same config, batch size 1, T 0; target-only parity = separate launch (not run)"}
    except Exception as _e:
        out["parity_control"] = {"error": repr(_e)[:300]}
    print("E12 rev 6 preflight:", json.dumps(out)[:800], flush=True)
    return out


def _e12_branch_done(rec):
    with _E12_LOCK:
        E12_RECORDS.setdefault(rec["task_id"], []).append(rec)
        E12_RUN["branches_done"] += 1
        E12_RUN["branches_accepted"] += int(bool(rec.get("accepted")))
    try:
        e12_write_submission()
    except Exception as _e:
        print("E12 submission write failed:", repr(_e)[:200], flush=True)


_E12_STOP = threading.Event()


def _e12_heartbeat(every=300):
    while not _E12_STOP.wait(every):
        snap = {r.get("name"): r.get("value") for r in metrics_snapshot() if isinstance(r, dict) and "name" in r}
        keep = {k: v for k, v in snap.items() if any(s in k for s in ("num_running_reqs", "num_queue_reqs", "gen_throughput",
                                                                        "token_usage", "cache_hit_rate"))}
        print(f"E12 [{(time.perf_counter() - NOTEBOOK_T0) / 60:.0f} min] branches done {E12_RUN['branches_done']}"
              f"/{len(TASK_IDS) * E12_BRANCHES}, accepted {E12_RUN['branches_accepted']} | {json.dumps(keep)[:400]}", flush=True)


if ENGINE_READY and E10_SANDBOX_SELFTEST.get("ok"):
    try:
        try:
            E12_RUN["model_id"] = json.loads(_urlreq.urlopen(SGL_URL + "/v1/models", timeout=30).read())["data"][0]["id"]
        except Exception as _e:
            E12_RUN["model_id"] = str(SGL_MODEL_PATH)
            print("E12: /v1/models failed, using the model path:", repr(_e)[:200], flush=True)
        E12_RUN["streams"] = min(int(CONCURRENCY), E12_STREAMS or int(CONCURRENCY))
        os.environ["ARC3_MAX_ACTIVE_STREAMS"] = str(E12_RUN["streams"])
        _end = HARD_STOP - E12_STOP_BEFORE_HARD_SEC
        if E12_SOLVE_WINDOW_SEC:
            _end = min(_end, time.perf_counter() + E12_SOLVE_WINDOW_SEC)
        if P.get("hidden_equiv"):   # gate 4b (user 2026-10-21): this run gets the hidden run's compute per task
            _ra = PROFILE_TABLE["RERUN_ALL"]
            _hid_end = _ra["time_budget_sec"] - _ra["hard_margin_sec"] - E12_STOP_BEFORE_HARD_SEC
            _start = time.perf_counter() - NOTEBOOK_T0
            _per, _ = _e12_g4b_window(_hid_end, _start, len(TASK_IDS))          # hidden test = 240 tasks
            E12_RUN["g4b"] = {"hidden_solve_end_sec": _hid_end, "solve_start_sec": round(_start), "hidden_tasks": 240,
                              "sec_per_task": round(_per, 1), "tasks": len(TASK_IDS), "window_sec": round(_per * len(TASK_IDS))}
            _end = min(_end, time.perf_counter() + _per * len(TASK_IDS))
            print("E12 gate 4b:", json.dumps(E12_RUN["g4b"]), flush=True)
        E12_RUN["deadline_in_sec"] = round(_end - time.perf_counter())
        E12_RUN["started_sec"] = round(time.perf_counter() - NOTEBOOK_T0)
        print("E12 solving:", len(E12_ORDER), "tasks x", E12_BRANCHES, "branches | streams", E12_RUN["streams"],
              "| model", E12_RUN["model_id"], "| stops in", E12_RUN["deadline_in_sec"], "s | order head", E12_ORDER[:5], flush=True)
        threading.Thread(target=_e12_heartbeat, daemon=True).start()
        # rev 6 R6-S5 (GPT plan review 2): per-stream allowance = 90 % of the KV pool / streams, so 10 resident histories
        # plus their reply reserve fit; the reserve itself follows each request's cap (R6-D1)
        _kv = int(ENGINE_INFO.get("kv_tokens") or 866744)
        E12_RUN["rev6_kv_tokens"] = _kv
        E12_RUN["rev6_window"] = min(131072, int(E12_R6["E12_R6_KV_FRACTION"] * _kv / max(1, E12_RUN["streams"])))
        os.environ["E12_DUCK_CONTEXT_WINDOW"] = str(E12_RUN["rev6_window"])
        E12_RUN["rev6_preflight"] = _e12_r6_preflight(E12_RUN["streams"])
        if E12_RUN["rev6_preflight"].get("isolation_state") == "INCONCLUSIVE":
            # H6-17 (GPT rev 6e audit 3; user 2026-10-22): a missing marker without a leak is retried once before the
            # 6-stream fail-safe; a FAIL never is
            _first = E12_RUN["rev6_preflight"]
            E12_RUN["rev6_preflight"] = _e12_r6_preflight(E12_RUN["streams"])
            E12_RUN["rev6_preflight"]["retry_of"] = {k: _first.get(k) for k in ("isolation_state", "isolation_own_marker",
                                                                                  "isolation_leaks")}
        # C6-2 (rev 6c audits): full concurrency only on an isolation PASS; FAIL / INCONCLUSIVE / error -> 6 streams
        _workers = E12A.workers_for_isolation(E12_RUN["rev6_preflight"].get("isolation_ok"), E12_RUN["streams"])
        if E12_RUN["rev6_preflight"].get("isolation_state") != "PASS":
            print("E12 rev 6 WARNING: isolation preflight", E12_RUN["rev6_preflight"].get("isolation_state"),
                  "-> workers", _workers, flush=True)
        E12_RUN["rev6_workers"] = _workers
        print("E12 rev 6: window", E12_RUN["rev6_window"], "| workers", _workers, "| first exposure",
              E12_R6["E12_R6_FIRST_EXPOSURE"], "| branch 1 view", E12_R6["E12_R6_B1_VIEW"], flush=True)
        if not RERUN and (E12_SOLVE_WINDOW_SEC or (P.get("hidden_equiv") and E12_RUN.get("g4b"))):
            # SA-1 + H6-6 (GPT rev 6e audit 2): every commit window (gate 4b and the plain solve window) starts after the
            # preflight, bounded by the hard stop; the per-task compute of gate 4b stays the hidden run's. The hidden
            # rerun keeps its absolute end (no window). H6-3: nothing else runs before solving
            _shift = (time.perf_counter() - NOTEBOOK_T0) - E12_RUN["started_sec"]     # started_sec: set when _end was fixed
            _end = _e12_shifted_end(_end, _shift, HARD_STOP - E12_STOP_BEFORE_HARD_SEC)
            E12_RUN["window_shift"] = {"shift_sec": round(_shift, 1),
                                       "window_start_sec": round(time.perf_counter() - NOTEBOOK_T0)}
            if E12_RUN.get("g4b"):
                E12_RUN["g4b"].update(window_start_sec=E12_RUN["window_shift"]["window_start_sec"], shift_sec=round(_shift, 1))
            E12_RUN["deadline_in_sec"] = round(_end - time.perf_counter())
            print("E12: the solve window starts after the preflight (shift", round(_shift, 1), "s), stops in",
                  E12_RUN["deadline_in_sec"], "s", flush=True)
        if P.get("parity"):          # R6-X1 diagnostic profile: no solving (the fallback submission stays)
            E12_RUN["parity"] = _e12_parity()
        else:
          E12A.solve_all_r6({t: CHALLENGES[t] for t in E12_ORDER}, E12_ORDER, out_dir=E12_OUT,
                          deadline=time.monotonic() + max(0.0, _end - time.perf_counter()), base_url=SGL_URL + "/v1",
                          model_id=E12_RUN["model_id"], branches=E12_BRANCHES, max_generated_tokens=E12_MAX_GEN_PER_BRANCH,
                          save_request_logs=E12_REQUEST_LOGS, on_branch_done=_e12_branch_done_r6, workers=_workers,
                          first_exposure_tokens=E12_R6["E12_R6_FIRST_EXPOSURE"] or None, b1_view=E12_R6["E12_R6_B1_VIEW"],
                          min_admit_sec=E12_R6["E12_R6_MIN_ADMIT_SEC"], capacity_probe=_e12_capacity_probe,
                          kv_admit_limit=E12_R6["E12_R6_KV_ADMIT"], kv_poll_sec=E12_R6["E12_R6_KV_POLL"],
                          min_chunk_tokens=E12_R6["E12_R6_MIN_CHUNK"],
                          fair_share=bool(E12_R6["E12_R6_FAIR_SHARE"]),
                          stage2_p90_cutoff=bool(E12_R6["E12_R6_P90_CUTOFF"]))
        E12_RUN["rev6_scheduler"] = dict(E12A.SCHED_LAST)
        # H6-3 (GPT rev 6e audits): the cache diagnostic runs AFTER solving, so the solver starts from the same server
        # state as the hidden rerun (which never runs it). Its time comes from what is left before the output margin;
        # the solve deadline never moves for it. Skipped when under 90 s are left
        if not RERUN and not P.get("parity") and E12_R6.get("E12_R6_CACHE_DIAG"):
            _left = HARD_STOP - E12_STOP_BEFORE_HARD_SEC - time.perf_counter() - 30
            if (E12_RUN.get("rev6_scheduler") or {}).get("stragglers"):      # rev 6g H7-4
                E12_RUN["cache_diag"] = {"skipped": "a solver straggler may still be sending requests"}
            elif _left >= 90:
                try:
                    E12_RUN["cache_diag"] = _e12_cache_diag(budget_sec=min(240.0, _left))
                    E12_RUN["cache_diag"]["ran"] = "after solving"
                except Exception as _e:
                    E12_RUN["cache_diag"] = {"error": repr(_e)[:300]}
            else:
                E12_RUN["cache_diag"] = {"skipped": "less than 90 s before the output margin", "left_sec": round(_left)}
    except BaseException:
        E12_RUN["error"] = traceback.format_exc()[-3000:]
        print("E12 solver error (submission.json keeps what was written):", E12_RUN["error"], flush=True)
else:
    E12_RUN["error"] = f"not run: ENGINE_READY={ENGINE_READY}, sandbox self-test={E10_SANDBOX_SELFTEST.get('ok')}"
    print("E12:", E12_RUN["error"], "-> CELL 0 fallback submission", flush=True)
_E12_STOP.set()
E12_RUN["finished_sec"] = round(time.perf_counter() - NOTEBOOK_T0)
try:
    e12_write_submission()
except Exception:
    print("E12 final submission write failed:", traceback.format_exc()[-1000:], flush=True)


In [ ]:
# ==================================================================== CELL 7 — E12: report, score (evaluation) and bundle
try:
    with _E12_LOCK:
        _recs = {t: list(v) for t, v in E12_RECORDS.items()}
    _gen = E12A.build_submission({t: CHALLENGES[t] for t in TASK_IDS}, _recs)
    write_json_atomic(OUT / "e12_generated_attempts.json", _gen)

    def _usage_summary(u):
        p = sum(int(x.get("prompt") or 0) for x in u); c = sum(int(x.get("cached") or 0) for x in u)
        return {"requests": len(u), "prompt_tokens": p, "cached_tokens": c, "cached_pct": round(100 * c / p, 1) if p else None,
                "completion_tokens": sum(int(x.get("completion") or 0) for x in u),
                "cached_pct_by_request": [round(100 * int(x.get("cached") or 0) / int(x["prompt"]), 1) if x.get("prompt") else None
                                          for x in u][:400]}
    branches = []
    for t in TASK_IDS:
        for r in _recs.get(t, []):
            subs = r.get("submissions") or []
            branches.append({"task": t, "branch": r.get("branch"), "accepted": bool(r.get("accepted")), "turns": r.get("turns"),
                             "failures": r.get("failures"), "note": r.get("note"), "generated_tokens": r.get("generated_tokens"),
                             "total_tokens": r.get("total_tokens"), "sec": round((r.get("t1") or 0) - (r.get("t0") or 0), 1),
                             "submissions": len(subs), "train_exact_per_submission": [sum(map(bool, s.get("train_exact") or []))
                                                                                       for s in subs],
                             "usage": _usage_summary(r.get("usage") or []),
                             # rev 5 metrics (absent before rev 5): harvested check() candidates and event counters
                             "checked_verified": len(r.get("checked") or []),
                             "checked_kept": sum(1 for s in (r.get("checked") or []) if s.get("accepted")),
                             "rev5_events": r.get("rev5_events"),
                             # rev 6 (absent before rev 6)
                             "view": r.get("view"), "status": r.get("status"), "runs": r.get("runs"),
                             "progress_best": (r.get("progress") or {}).get("best"), "rev6_events": r.get("rev6_events"),
                             "stage2_class": r.get("stage2_class"),
                             "overfit_flags": sorted({f for v in (r.get("checked") or []) + (r.get("submissions") or [])
                                                      if v.get("accepted") for f in (v.get("overfit_flags") or [])})})
    _all_u = [x for t in TASK_IDS for r in _recs.get(t, []) for x in (r.get("usage") or [])]
    report = {"run_id": RUN_ID, "profile": PROFILE, "rerun": RERUN, "configuration": CONFIG, "engine": ENGINE_INFO,
              "e12_run": E12_RUN, "order": E12_ORDER, "branches": branches, "usage_all": {k: v for k, v in _usage_summary(_all_u).items()
                                                                                        if k != "cached_pct_by_request"},
              "tasks_planned": len(TASK_IDS), "queries_planned": sum(len(CHALLENGES[t]["test"]) for t in TASK_IDS)}
    def _r6_metrics(rows):   # R6-M1: one report contract (censored first-check = the branch total)
        import statistics as _st
        med = lambda v: _st.median(v) if v else None
        ev = [r.get("rev6_events") or {} for r in rows]
        tot = lambda k: sum(int(e.get(k) or 0) for e in ev)
        ft = [e.get("first_tool_tokens") for e in ev if e.get("first_tool_tokens") is not None]
        fc = [(e.get("first_check_tokens") if e.get("first_check_tokens") is not None else r.get("generated_tokens") or 0)
              for e, r in zip(ev, rows)]
        hist = lambda k: {kk: sum(int((e.get(k) or {}).get(kk, 0)) for e in ev)
                          for kk in sorted({x for e in ev for x in (e.get(k) or {})})}
        vs = sorted(x for e in ev for x in (e.get("verify_sec") or []))
        b0 = [r for r in rows if r.get("branch") == 0]
        return {"branches": len(rows),
                "tasks_exposed": len({r.get("task") for r in b0 if (r.get("generated_tokens") or 0) > 0}),
                "tasks_meaningful": len({r.get("task") for r in rows
                                         if (r.get("rev6_events") or {}).get("first_tool_tokens") is not None}),
                "first_tool_tokens_median": med(ft), "first_check_tokens_median_censored": med(fc),
                # G4-13 (GPT G4B audit 1 M1-B): the observed median is over checked branches only (selection bias)
                "first_check_tokens_median_observed": med([e["first_check_tokens"] for e in ev
                                                           if e.get("first_check_tokens") is not None]),
                "first_check_tokens_median_observed_b0": med([e["first_check_tokens"] for e, r in zip(ev, rows)
                                                              if e.get("first_check_tokens") is not None and r.get("branch") == 0]),
                "checked_branches": sum(1 for e in ev if e.get("first_check_tokens") is not None),
                "never_checked": sum(1 for e in ev if e.get("first_check_tokens") is None),
                "autocheck_calls": tot("autocheck_calls"), "autocheck_functions": tot("autocheck_functions"),
                "grid_rows_suppressed": tot("grid_rows_suppressed"), "length_cut_no_call": tot("length_cut_no_call"),
                "nudges_replaced": tot("nudges_replaced"), "timeouts_clamped": tot("timeouts_clamped"),
                "verify_skipped_deadline": tot("verify_skipped_deadline"),
                "accepted_duplicate_outputs": tot("accepted_duplicate_outputs"),
                "resumes": hist("resumes"), "caps": hist("caps"),
                "verify_sec_p50": vs[len(vs) // 2] if vs else None, "verify_sec_p90": vs[int(len(vs) * 0.9)] if vs else None,
                "status": {s: sum(1 for r in rows if r.get("status") == s) for s in ("paused", "done", "new")},
                **_r6_metrics_more([dict(r, task=t) for t in TASK_IDS for r in _recs.get(t, [])])}

    def _r6_metrics_more(rows):   # rev 6b additions (fix-list R6-M1; raw branch records; label-free, no truth)
        ev = [r.get("rev6_events") or {} for r in rows]
        q = lambda v, f: sorted(v)[min(len(v) - 1, int(f * len(v)))] if v else None
        tot = lambda k: sum(int(e.get(k) or 0) for e in ev)
        rc = tot("reasoning_chars")
        ft = [e.get("facts_tokens") for e in ev if e.get("facts_tokens") is not None]
        acc = [(r, v) for r in rows for v in (r.get("checked") or []) + (r.get("submissions") or []) if v.get("accepted")]
        # G4-7 / G4-13 (GPT G4B audits): a check() and a submit() of one program are 2 events, 1 program
        uniq, preds, progs = {}, {}, {}
        for r, v in acc:
            uniq.setdefault((r.get("task"), r.get("branch"), v.get("name"), json.dumps(v.get("test_outputs"))), v)
            preds.setdefault(r.get("task"), set()).add(json.dumps(v.get("test_outputs")))
            # H6-9 (GPT rev 6e audits 2/3): a program = its source + retained dependencies (src_key), across branches
            progs.setdefault(r.get("task"), set()).add(v.get("src_key") or (r.get("branch"), v.get("name"),
                                                                             json.dumps(v.get("test_outputs"))))
        flags = {}
        for v in uniq.values():
            for f in v.get("overfit_flags") or []:
                flags[f] = flags.get(f, 0) + 1
        by_task = {}
        for r, v in acc:
            by_task.setdefault(r.get("task"), {}).setdefault(r.get("branch"), set()).add(json.dumps(v.get("test_outputs")))
        both = [t for t, d in by_task.items() if len(d) >= 2]
        agree = sum(1 for t in both if set.intersection(*by_task[t].values()))
        b1 = {}
        for r in rows:
            if r.get("branch") == 1 and r.get("stage2_class") is not None:
                own = by_task.get(r.get("task"), {})
                new = len(own.get(1, set()) - own.get(0, set()))
                c = b1.setdefault(str(r["stage2_class"]), {"branches": 0, "new_distinct_verified": 0, "tokens": 0})
                c["branches"] += 1
                c["new_distinct_verified"] += new
                c["tokens"] += int(r.get("generated_tokens") or 0)
        runs = [x for r in rows for x in (r.get("runs") or [])]
        osh = [x["overshoot"] for x in runs if "overshoot" in x]
        rp = [x["first_uncached"] for x in runs if x.get("stage") == 2 and "first_uncached" in x]
        pe = [a / b for r in rows for a, b in ((r.get("rev5_events") or {}).get("prompt_estimates") or []) if b]
        us = [u for r in rows for u in (r.get("usage") or [])]
        try:
            dec = e11_decode_summary() or {}
        except Exception:
            dec = {}
        rh = dec.get("running_hist") or {}
        return {"facts_tokens_p50": q(ft, .5), "facts_tokens_p90": q(ft, .9), "facts_tokens_max": max(ft) if ft else None,
                "hand_reading_grid_row_share": round(tot("reasoning_grid_row_chars") / rc, 4) if rc else None,
                "reasoning_digit_share": round(tot("reasoning_digit_chars") / rc, 4) if rc else None,
                "reasoning_repeat_share": round(tot("reasoning_repeat_chars") / rc, 4) if rc else None,
                "tool_calls": tot("tool_calls"), "redefinition_share": round(tot("redefinitions") / max(1, tot("tool_calls")), 4),
                "infra_errors": tot("infra_errors"), "trim_reasoning_only_blocks": tot("trim_reasoning_only_blocks"),
                "facts_over_target": sum(1 for e in ev if e.get("facts_over_target")),
                "accepted_events": len(acc), "accepted_candidates": len(uniq), "unique_verified_programs": len(uniq),
                # H7-5: the field above is keyed (task, branch, name, outputs), i.e. name / output candidates; the program
                # count by source + dependencies is verified_programs_by_source
                "unique_name_output_candidates": len(uniq),
                "distinct_test_predictions": sum(len(v) for v in preds.values()),
                # H6-9: the old field counted distinct predictions; both are reported now
                "verified_programs_by_source": sum(len(v) for v in progs.values()),
                "verified_programs_per_task": {t: len(v) for t, v in sorted(progs.items())},
                "distinct_predictions_per_task": {t: len(v) for t, v in sorted(preds.items())},
                "overfit_flags": flags, "overfit_flags_counted_on": "unique programs",
                "tasks_with_two_branch_candidates": len(both), "tasks_where_branches_agree": agree,
                "branch1_yield_by_stage2_class": b1,
                "overshoot_p50": q(osh, .5), "overshoot_max": max(osh) if osh else None,
                "stage2_resume_uncached_p50": q(rp, .5), "stage2_resume_uncached_p90": q(rp, .9),
                "prompt_estimate_over_actual_p50": round(q(pe, .5), 3) if pe else None,
                "uncached_prompt_tokens": sum(int(u.get("prompt") or 0) - int(u.get("cached") or 0) for u in us),
                "generated_tokens": sum(int(r.get("generated_tokens") or 0) for r in rows),
                "real_running_max": max((int(k) for k, v in rh.items() if v), default=None) if rh else None,
                "accept_len_median": dec.get("accept_len_median"),
                "fair_share_last": ((E12_RUN.get("rev6_scheduler") or {}).get("shares") or [None])[-1],
                **_r6e_metrics(rows, ev, b1, us)}

    def _r6e_metrics(rows, ev, b1, us):   # rev 6e: G4-14 / G4-15 / G4-18 (label-free, no truth)
        import statistics as _st
        med = lambda v: round(_st.median(v), 1) if v else None
        n = len(rows)
        ftt = [e.get("first_transform_tokens") for e in ev]
        obs = [x for x in ftt if x is not None]
        cens = [x if x is not None else int(r.get("generated_tokens") or 0) for x, r in zip(ftt, rows)]
        best = {}
        for r in rows:
            b = ((r.get("progress") or {}).get("best") or {})
            k = f"{b.get('k')}/{b.get('n')}" if b else "none"
            best[k] = best.get(k, 0) + 1
        b1_tok = sum(c["tokens"] for c in b1.values())
        b1_new = sum(c["new_distinct_verified"] for c in b1.values())
        reqs = [q for r in rows for q in (r.get("requests") or [])]
        later = [q for q in reqs if q.get("i", 0) > 0 and q.get("prompt")]
        zero = [q for q in later if not q.get("cached")]
        runs2 = [x for r in rows for x in (r.get("runs") or []) if x.get("stage") == 2]

        def _yield(sel):   # H6-14 / H7-3: new verified predictions per 100k generated tokens (all / b0 / b1, by stage)
            rr = [(x, r) for r in rows for x in (r.get("runs") or []) if sel(x, r)]
            tok = sum(int(x.get("tokens") or 0) for x, _ in rr)
            nb = sum(int(x.get("new_in_branch") or 0) for x, _ in rr)
            nt = sum(int(x.get("new_for_task") or 0) for x, _ in rr)
            return {"runs": len(rr), "tokens": tok, "new_in_branch": nb, "new_for_task": nt,
                    "per_100k_for_task": round(1e5 * nt / tok, 3) if tok else None,
                    "per_100k_in_branch": round(1e5 * nb / tok, 3) if tok else None}
        nud = [(e, r) for e, r in zip(ev, rows) if e.get("provisional_nudge_tokens") is not None]
        gap = lambda k: [e[k] - e["provisional_nudge_tokens"] for e, _ in nud if e.get(k) is not None]
        return {
            # H6-14 (GPT rev 6e audits): the allocation question is decided on these
            "verified_yield": {"all": _yield(lambda x, r: True), "b0": _yield(lambda x, r: r.get("branch") == 0),
                               "b1": _yield(lambda x, r: r.get("branch") == 1),
                               "stage1": _yield(lambda x, r: x.get("stage") == 1),
                               "stage2": _yield(lambda x, r: x.get("stage") == 2)},
            # H6-18: nudge efficacy (H6-1: sent in a tool result at 24k, else in an opener)
            "nudge": {"branches": len(nud), "via": {k: sum(1 for e, _ in nud if e.get("provisional_nudge_via") == k)
                                                     for k in ("tool", "opener")},
                      # H7-10: emitted (above) vs seen by the model in a later request
                      "seen": sum(1 for e, _ in nud if e.get("provisional_nudge_seen_request") is not None),
                      "emitted_not_seen": sum(1 for e, _ in nud if e.get("provisional_nudge_seen_request") is None),
                      "tokens_p50": med([e["provisional_nudge_tokens"] for e, _ in nud]),
                      "transform_after": len(gap("first_transform_after_nudge")),
                      "tokens_to_transform_p50": med(gap("first_transform_after_nudge")),
                      "check_after": len(gap("first_check_after_nudge")),
                      "tokens_to_check_p50": med(gap("first_check_after_nudge")),
                      "accepted": sum(1 for _, r in nud if r.get("accepted")
                                      or any(v.get("accepted") for v in (r.get("checked") or [])))},
            "progress_same_name_new_source": sum(int(e.get("progress_same_name_new_source") or 0) for e in ev),
            "indexed_view_over_target": sum(1 for e in ev if e.get("view_over_target")),             # H7-8
            "tasks_with_response": len({r.get("task") for r in rows if r.get("branch") == 0
                                        and (r.get("generated_tokens") or 0) > 0}),                  # H7-7
            # G4-14
            "first_transform_tokens_median_observed": med(obs), "first_transform_tokens_median_censored": med(cens),
            "branches_with_transform": len(obs), "branches_with_transform_within_20k": sum(1 for x in obs if x <= 20000),
            "share_with_transform_within_20k": round(sum(1 for x in obs if x <= 20000) / n, 3) if n else None,
            "best_single_function_pairs": dict(sorted(best.items())),
            "b1_new_distinct_verified_per_100k_tokens": round(1e5 * b1_new / b1_tok, 3) if b1_tok else None,
            "provisional_nudges": sum(1 for e in ev if e.get("provisional_nudge_tokens") is not None),
            # G4-18
            "progress_reports": sum(int(e.get("progress_reports") or 0) for e in ev),
            "progress_same_improved": sum(int(e.get("progress_same_improved") or 0) for e in ev),
            "progress_new_hypothesis": sum(int(e.get("progress_new_hypothesis") or 0) for e in ev),
            "stage2_runs": len(runs2), "stage2_runs_improved": sum(1 for x in runs2 if x.get("improved")),
            # G4-15 (null prompt_tokens_details = 0 cached: SGLang 0.5.21 omits it when nothing was cached)
            "requests_instrumented": len(reqs), "later_requests": len(later), "later_zero_hit": len(zero),
            "later_zero_hit_prefix_intact": sum(1 for q in zero if q.get("prefix_ok")),
            "later_zero_hit_trimmed": sum(1 for q in zero if q.get("trimmed")),
            "later_zero_hit_idle_sec_p50": med([q["idle_sec"] for q in zero if q.get("idle_sec") is not None]),
            "later_idle_sec_p50": med([q["idle_sec"] for q in later if q.get("idle_sec") is not None]),
            "later_zero_hit_mamba_p50": med([q["mamba"] for q in zero if q.get("mamba") is not None]),
            "later_prefix_intact_share": round(sum(1 for q in later if q.get("prefix_ok")) / len(later), 3) if later else None,
            # H6-10: character-prefix share and tools / template changes for the zero-hit later requests (still a proxy)
            "later_zero_hit_char_prefix_full": sum(1 for q in zero if q.get("serialized_char_prefix") is not None
                                                   and q.get("prev_chars") and q["serialized_char_prefix"] >= q["prev_chars"] - 2),
            # H7-13 (GPT rev 6f audit 3): unexplained = the serialized prompt kept its prefix, tools / template unchanged,
            # not trimmed, and still 0 cached; otherwise the prompt changed (reason). A proxy, never proof of eviction
            "later_zero_hit_class": _zero_classes(zero),
            "later_zero_hit_tools_or_template_changed": sum(1 for q in zero if q.get("tools_same") is False
                                                            or q.get("template_same") is False),
            "cached_not_reported_requests": sum(1 for q in reqs if q.get("cached_reported") is False),
            "cache_null_detail_requests": sum(1 for u in us if u.get("cached") is None),
            "cache_note": "aggregate over all requests; a missing prompt_tokens_details means 0 cached (SGLang 0.5.21)",
            "concurrency": _r6e_concurrency()}

    def _zero_classes(zero):   # H7-13
        out = {}
        for q in zero:
            full = (q.get("serialized_char_prefix") is not None and q.get("prev_chars")
                    and q["serialized_char_prefix"] >= q["prev_chars"] - 2)
            if q.get("trimmed"):
                k = "prompt_changed:trimmed"
            elif q.get("tools_same") is False:
                k = "prompt_changed:tools"
            elif q.get("template_same") is False:
                k = "prompt_changed:template"
            elif not full:
                k = "prompt_changed:prefix"
            else:
                k = "unexplained_cache_miss"
            out[k] = out.get(k, 0) + 1
        return out

    def _r6e_concurrency():   # G4-15 / Q5: decode rate by running count, split by "a prefill batch ran just before"
        import re as _re, statistics as _st
        try:
            text = E11_SERVER.log_text() if E11_SERVER is not None else ""
        except Exception:
            return None
        g = lambda pat, ln: (lambda m: float(m.group(1)) if m else None)(_re.search(pat, ln))
        by, after = {}, False
        for ln in text.splitlines():
            if "TP0]" not in ln:
                continue
            if "Prefill batch" in ln:
                after = True
                continue
            if "Decode batch" not in ln:
                continue
            n = g(r"#running-req: (\d+)", ln)
            row = by.setdefault(str(int(n)) if n is not None else "?", {"lines": 0, "tok": [], "tok_after_prefill": [],
                                                                         "accept": [], "mamba": []})
            row["lines"] += 1
            t = g(r"gen throughput \(token/s\): ([\d.]+)", ln)
            if t is not None:
                (row["tok_after_prefill"] if after else row["tok"]).append(t)
            for k, pat in (("accept", r"accept len: ([\d.]+)"), ("mamba", r"mamba usage: ([\d.]+)")):
                v = g(pat, ln)
                if v is not None:
                    row[k].append(v)
            after = False
        m = lambda v: round(_st.median(v), 2) if v else None
        return {k: {"lines": v["lines"], "tok_s_p50_clean": m(v["tok"]), "tok_s_p50_after_prefill": m(v["tok_after_prefill"]),
                    "share_after_prefill": round(len(v["tok_after_prefill"]) / max(1, len(v["tok"]) + len(v["tok_after_prefill"])), 3),
                    "accept_len_p50": m(v["accept"]), "mamba_usage_p50": m(v["mamba"])}
                for k, v in sorted(by.items(), key=lambda kv: (len(kv[0]), kv[0]))}
    if _E12_REV6:
        report["rev6"] = {"metrics": _r6_metrics(branches), "scheduler": E12_RUN.get("rev6_scheduler"),
                          "window": E12_RUN.get("rev6_window"), "workers": E12_RUN.get("rev6_workers"),
                          "preflight": E12_RUN.get("rev6_preflight"), "g4b": E12_RUN.get("g4b"),
                          "parity": E12_RUN.get("parity"), "cache_diag": E12_RUN.get("cache_diag")}
    if SOLUTIONS is not None:
        hits = pass1 = pass2 = total = 0; task_norm = 0.0; recs_eval = []
        for t in TASK_IDS:
            th = 0
            for qi, att in enumerate(_gen[t]):
                gold = SOLUTIONS[t][qi]; a = att["attempt_1"] == gold; b = att["attempt_2"] == gold
                hits += a or b; pass1 += a; pass2 += b; total += 1; th += a or b
                acc = [s for r in _recs.get(t, []) for s in (r.get("submissions") or []) if s.get("accepted")]
                harv = [s for r in _recs.get(t, []) for s in (r.get("checked") or []) if s.get("accepted")]
                recs_eval.append({"task": t, "query": qi, "hit": bool(a or b), "pass1": bool(a), "pass2": bool(b),
                                  "accepted_programs": len(acc),
                                  "accepted_programs_correct": sum(1 for s in acc if qi < len(s.get("test_outputs") or [])
                                                                   and s["test_outputs"][qi] == gold),
                                  "harvested_programs": len(harv),
                                  "harvested_programs_correct": sum(1 for s in harv if qi < len(s.get("test_outputs") or [])
                                                                    and s["test_outputs"][qi] == gold)})
            task_norm += th / max(1, len(_gen[t]))
        report.update(eval_query_hits=hits, eval_query_total=total, eval_pass1_hits=pass1, eval_pass2_hits=pass2,
                      eval_task_norm_pct=round(100 * task_norm / max(1, len(TASK_IDS)), 2), eval_unit_records=recs_eval)
    try:
        report["e11"] = {"decode_summary": e11_decode_summary(),
                         "shim": engine.stats if isinstance(getattr(engine, "stats", None), dict) else None,
                         "server_log": str(E11_SERVER.log_path) if E11_SERVER is not None else None}
    except Exception as _e11e:
        report["e11"] = {"error": repr(_e11e)[:300]}
    if _E12_REV6 and isinstance(report.get("rev6"), dict):      # rev 6g H7-12 (GPT rev 6f audit 2): per GPU-hour
        try:
            _gh = 4 * max(1, (E12_RUN.get("finished_sec") or 0) - (E12_RUN.get("started_sec") or 0)) / 3600.0
            _solved = len({u["task"] for u in (report.get("eval_unit_records") or [])}
                          - {u["task"] for u in (report.get("eval_unit_records") or []) if not u.get("hit")})
            _yt = (((report["rev6"].get("metrics") or {}).get("verified_yield") or {}).get("all") or {}).get("new_for_task")
            report["rev6"]["per_gpu_hour"] = {
                "gpu_hours": round(_gh, 3),
                "correct_inputs": round(report["eval_query_hits"] / _gh, 3) if report.get("eval_query_hits") is not None else None,
                "tasks_solved": round(_solved / _gh, 3) if report.get("eval_unit_records") else None,
                "new_for_task_predictions": round(_yt / _gh, 3) if _yt is not None else None}
        except Exception as _e:
            report["rev6"]["per_gpu_hour"] = {"error": repr(_e)[:200]}
    write_json_atomic(OUT / "e12_run_report.json", report)
    if _E12_REV6:
        print("E12 rev 6 metrics:", json.dumps(report["rev6"]["metrics"])[:1500], flush=True)
        _sc = report["rev6"].get("scheduler") or {}
        # A3-F4 (rev 6c audits): full coverage is claimed only when stage1_left == 0
        print("E12 rev 6 coverage: tasks initialized", _sc.get("tasks_initialized"), "| request sent",
              _sc.get("tasks_with_request_sent"), "| response", (report["rev6"].get("metrics") or {}).get("tasks_with_response"),
              "/", len(TASK_IDS), "| stage-1 starts left",
              _sc.get("stage1_left"), "| workers", report["rev6"].get("workers"), "(fixed)", "| KV holds", _sc.get("kv_holds"),
              "for", _sc.get("kv_hold_sec"), "s | idle overrides", _sc.get("kv_idle_overrides"),
              "| KV samples", _sc.get("kv_samples"), "missing", _sc.get("kv_missing"),
              "| stragglers", _sc.get("stragglers"), flush=True)
    print("E12 RUN REPORT:", OUT / "e12_run_report.json", "| eval", report.get("eval_query_hits"), "/", report.get("eval_query_total"),
          "| task-norm", report.get("eval_task_norm_pct"), "| accepted branches", E12_RUN["branches_accepted"],
          "| cached", report["usage_all"].get("cached_pct"), "%", flush=True)
    e12_write_submission()
    _pack = WORKING / "Q27X_E12_DUCK_BUNDLE.zip"
    with zipfile.ZipFile(_pack, "w", zipfile.ZIP_DEFLATED) as z:
        for fp in (SUBMISSION_PATH, OUT / "e12_run_report.json", OUT / "e12_generated_attempts.json",
                   OUT / "sglang_decode_summary.json", OUT / "sglang_requests.jsonl"):
            if fp.exists():
                z.write(fp, arcname=fp.name)
        for fp in sorted(E12_OUT.rglob("*")):     # transcripts (full thinking) + branch results
            if fp.is_file() and (fp.name in ("transcript.txt", "branch_result.json") or fp.name.endswith("requests.jsonl")):
                z.write(fp, arcname="duck/" + fp.relative_to(E12_OUT).as_posix())
        if E11_SERVER is not None and E11_SERVER.log_path.exists():
            z.write(E11_SERVER.log_path, arcname=E11_SERVER.log_path.name)
    print("E12 bundle:", _pack, round(_pack.stat().st_size / 1e6, 1), "MB", flush=True)
except BaseException:
    print("E12 reporting error (submission.json still exists):", traceback.format_exc()[-3000:], flush=True)
try:
    e11_kill_server(wait=False)
except Exception:
    pass


In [ ]:
# ==================================================================== CELL 6b — E14-A2: more engine arms after the Duck run
# Plan: Notes/13. The Duck run's server was stopped by CELL 7. Phases (each its own launch): the Duck arm again with an
# agent-like replay (Mamba checkpoint eviction, E14-M) and a cache-hit parity pass; + int8 Mamba checkpoints (same
# replay + the cache-hit parity); target-only (DFlash correctness); DFlash width 6 and 4; the NCCL env arm if the bench
# found one >= 20 % faster at 983,040 B. Nothing here changes the submission.
if E14_DO_MEASURE and E14_CTX is not None:
    E14_RESERVE = 0.0
    _S = E14_PROMPTS["short"]
    _sess = [(s, u) for s, u, _t in E14_PROMPTS["duck"]] * 2
    _base_arm = dict(E14_KEPT)          # the adopted arm (the Duck may have run on another one in a failure path)

    def _cache_hit_parity(tag):
        globals()["E14_LAST_FLUSH"] = E14M.flush_cache(E14_CTX)     # review 9: these prompts were sent already
        a = E14M.parity_pass(E14_CTX, _S[:6], max_new=384, tag=tag + "_cold")
        b = E14M.parity_pass(E14_CTX, _S[:6], max_new=384, tag=tag + "_cached")
        return E14M.compare_passes(a, b, tag + ": fresh vs cached prefix")

    def _a2_reference():
        """The matched width-8 control (audit 18 E14-8): greedy rows of the arm the Duck ran, on THIS launch; every A2
        arm gate below compares with these (int8 cross-arm, target-only, widths, NCCL)."""
        a = E14M.parity_pass(E14_CTX, _S[:6], max_new=256, tag="a2ref_serial_a")
        b = E14M.parity_pass(E14_CTX, _S[:6], max_new=256, tag="a2ref_serial_b")
        c = E14M.parity_pass(E14_CTX, _S, max_new=256, concurrent=True, tag="a2ref_conc_a")
        c2 = E14M.parity_pass(E14_CTX, _S, max_new=256, concurrent=True, tag="a2ref_conc_b")
        globals()["E14_A2_REF"] = E14M.gate_reference(a, b, c, c2)
        return {"serial_aa": E14M.compare_passes(a, b, "A2 base serial A/A"),
                "conc_aa": E14M.compare_passes(c, c2, "A2 base all-at-once A/A")}

    E14_A2_REF = None
    _gate = lambda tag, identity=False: E14M.arm_gate(E14_CTX, _S, E14_A2_REF, tag="gate_" + tag, identity=identity)

    def _gated(tag, windows):
        """The gate first; the throughput windows only when it passed (correctness before speed)."""
        out = {"gate": _gate(tag)}
        if not out["gate"].get("ok"):
            out["windows_skipped"] = "the correctness gate did not pass"
            return out
        out.update(windows())
        return out
    _ok, _ = e14_launch("a2_duck_arm", set_=_base_arm["set"], drop=_base_arm["drop"], env=_base_arm["env"])
    e14_phase("A2-0_reference", _a2_reference, _ts(8 * 60)) if _ok else None
    _r0 = e14_phase("A2-1_agent_replay_default", lambda: {
        "replay": E14M.agent_replay(E14_CTX, _sess, turns=6, admit=SGL_MAX_RUNNING, tool_text=E14_PROMPTS["tool_text"],
                                    tag="default"),
        "cache_parity": _cache_hit_parity("default")}, _ts(14 * 60)) if _ok else None
    _ok, _ = e14_launch("int8_mamba", set_=dict(_base_arm["set"], **{"--enable-int8-mamba-checkpoint": None}),
                        drop=_base_arm["drop"], env=_base_arm["env"])
    _r1 = e14_phase("A2-2_int8_mamba", lambda: _gated("int8", lambda: {   # cross-arm: default vs int8, same ids
        "replay": E14M.agent_replay(E14_CTX, _sess, turns=6, admit=SGL_MAX_RUNNING, tool_text=E14_PROMPTS["tool_text"],
                                    tag="int8"),
        "cache_parity": _cache_hit_parity("int8")}), _ts(16 * 60)) if _ok else None
    try:
        _c0 = ((_r0 or {}).get("replay") or {}).get("classes") or {}
        _c1 = ((_r1 or {}).get("replay") or {}).get("classes") or {}
        _miss = lambda c: (c.get("deeper_miss", 0) + c.get("lost_at_prefill_end", 0)) / max(1, sum(c.values()))
        _pa = E14M.parity_ok((_r1 or {}).get("cache_parity"), (_r0 or {}).get("cache_parity"))
        e14_decide("int8_mamba", [("measured", bool(_r1 and _r0)),
                                  ("cross-arm correctness gate", ((_r1 or {}).get("gate") or {}).get("ok")),
                                  ("fewer cache misses", bool(_c1) and _miss(_c1) < _miss(_c0)),
                                  ("cache-hit parity", _pa["ok"]),
                                  ("no server error signs", not E14M.log_errors((_r1 or {}).get("server_log_signs")))],
                   miss_share_default=round(_miss(_c0), 3), miss_share_int8=round(_miss(_c1), 3), cache_parity=_pa,
                   gate=(_r1 or {}).get("gate"))
    except Exception as _e:
        E14_RESULTS["decisions"]["int8_mamba"] = {"adopt": False, "error": repr(_e)[:200]}

    _ok, _ = e14_launch("target_only", set_=_base_arm["set"], drop=_base_arm["drop"], env=_base_arm["env"], spec=False)
    _rt = e14_phase("A2-3_target_only", lambda: _gated("target", lambda: {
        "short": E14M.steady_window(E14_CTX, _S, level=SGL_MAX_RUNNING, secs=_ts(60), tag="tonly", spec=False)}),
        _ts(14 * 60)) if _ok else None
    E14_RESULTS["decisions"]["dflash_vs_target"] = ((_rt or {}).get("gate") or {"ok": False, "why": "not run"})

    _ref_short = E14_KEPT_WIN.get("short")      # the kept arm's window = the arm the Duck run used
    for _w in (6, 4):
        _ok, _ = e14_launch(f"width{_w}", set_=dict(_base_arm["set"], **{"--speculative-num-draft-tokens": _w}),
                            drop=_base_arm["drop"], env=_base_arm["env"])
        _rw = e14_phase({6: "A2-4_width6", 4: "A2-5_width4"}[_w], lambda w=_w: _gated(f"w{w}", lambda: {
            "short": E14M.steady_window(E14_CTX, _S, level=SGL_MAX_RUNNING, secs=_ts(75), tag=f"w{w}short")}),
            _ts(12 * 60)) if _ok else None
        _sp = E14M.adopt_speed(_ref_short, (_rw or {}).get("short"), min_gain=0.05)
        _acc = ((_rw or {}).get("short") or {}).get("accept_p50")
        e14_decide(f"width{_w}", [("measured", bool(_rw)),
                                  ("correctness gate", ((_rw or {}).get("gate") or {}).get("ok")),
                                  ("speed >= +5 %", _sp["adopt"]),
                                  ("accept <= width", _acc is not None and _acc <= _w),
                                  ("no server error signs", not E14M.log_errors((_rw or {}).get("server_log_signs")))],
                   speed=_sp, accept_p50=_acc, gate=(_rw or {}).get("gate"))
    # NCCL env arm: only if the bench found an arm >= 20 % faster per call at 983,040 B inside CUDA graphs (eager AND
    # graph results correct); adopted only with a real-window gain and a passing correctness gate (transport safety)
    _nb = (E14_RESULTS.get("nccl") or {}).get("summary") or {}
    _win = _nb.get("adopt_candidate")
    _envs = {"p2p_sys": {"NCCL_P2P_LEVEL": "SYS"}, "shm_memcpy": {"NCCL_SHM_USE_CUDA_MEMCPY": "1"}}
    if _win in _envs:
        _ok, _ = e14_launch("nccl_" + _win, set_=_base_arm["set"], drop=_base_arm["drop"],
                            env=dict(_base_arm["env"], **_envs[_win]))
        _rn = e14_phase("A2-6_nccl_" + _win, lambda: _gated("nccl", lambda: {
            "short": E14M.steady_window(E14_CTX, _S, level=SGL_MAX_RUNNING, secs=_ts(75), tag="nccl")}),
            _ts(12 * 60)) if _ok else None
        _sp = E14M.adopt_speed(_ref_short, (_rn or {}).get("short"), min_gain=0.05, per_step=True)
        e14_decide("nccl_" + _win, [("measured", bool(_rn)),
                                    ("correctness gate", ((_rn or {}).get("gate") or {}).get("ok")),
                                    ("speed >= +5 %", _sp["adopt"]),
                                    ("no server error signs", not E14M.log_errors((_rn or {}).get("server_log_signs")))],
                   speed=_sp, gate=(_rn or {}).get("gate"))
    else:
        E14_RESULTS["decisions"]["nccl_env"] = {"adopt": False, "why": "no bench arm >= 20 % faster at 983,040 B",
                                                "bench_summary": _nb}
    try:
        e11_kill_server(wait=True)
    except Exception:
        pass
    E14_STATE.enter("A2_DONE")
    e14_save()


In [ ]:
# ==================================================================== CELL 7b — E14: report, JIT cache, bundle
try:
    E14_STATE.enter("REPORT")
    if E14_DO_MEASURE:
        E14_RESULTS["duck_run"] = {"e12_run": E12_RUN, "usage_all": (report or {}).get("usage_all") if "report" in globals() else None,
                                   "eval": {k: (report or {}).get(k) for k in ("eval_query_hits", "eval_query_total",
                                                                                "eval_task_norm_pct")} if "report" in globals() else None,
                                   "raw_reasoning": [r.get("raw_reasoning") for v in E12_RECORDS.values() for r in v]}
    # JIT / compile caches for the next runs (Notes/07, 11): nv/ and cuda/ never shipped; a 13-field manifest
    if E14_DO_MEASURE and E14_VENV.get("vpy") and not E11_HARNESS and HARD_STOP - time.perf_counter() > 600:
        try:
            _q = subprocess.run([E14_VENV["vpy"], "-c", "import importlib.metadata as m, json, torch; print(json.dumps({k: m.version(k) "
                                 "for k in ('sglang', 'sgl-kernel', 'flashinfer-python', 'torch', 'triton')} | "
                                 "{'cuda_runtime': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), "
                                 "'sm': '.'.join(map(str, torch.cuda.get_device_capability(0)))}))"],
                                capture_output=True, text=True, timeout=180, env=E14_VENV["env"])
            _v = json.loads(_q.stdout.strip().splitlines()[-1])
            _nv = subprocess.run([str(Path(E14_VENV["env"]["CUDA_HOME"]) / "bin" / "nvcc"), "--version"], capture_output=True,
                                 text=True, timeout=60).stdout.strip().splitlines()[-1:]
            _drv = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], capture_output=True,
                                  text=True, timeout=60).stdout.strip().splitlines()[:1]
            _site = Path(subprocess.run([E14_VENV["vpy"], "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"],
                                        capture_output=True, text=True, timeout=60).stdout.strip())
            _man = E14L.jit_manifest(sglang=_v["sglang"], sgl_kernel=_v["sgl-kernel"], flashinfer=_v["flashinfer-python"],
                                     torch=_v["torch"], triton=_v["triton"], cuda_runtime=_v["cuda_runtime"],
                                     nvcc=" ".join(_nv) or "unknown", driver=" ".join(_drv) or "unknown", gpu_name=_v["gpu"],
                                     sm=_v["sm"], wheels=E14L.wheel_inventory(SGL_WHEELHOUSE),
                                     jit_sources=E14L.source_fingerprint([_site / "sglang" / "jit_kernel", _site / "sgl_kernel",
                                                                          _site / "flashinfer"]),
                                     env=E14L.kernel_env(E14_VENV["env"]))
            _tar = E14L.save_jit_cache("/tmp/e11-cache", WORKING / "e14_jit_cache.tar", _man)
            E14_RESULTS["jit_cache"] = {"tar": str(_tar), "bytes": _tar.stat().st_size, "manifest": _man}
        except Exception as _e:
            E14_RESULTS["jit_cache"] = {"error": repr(_e)[:300]}
    E14_STATE.end("done")
    e14_save()
    _lines = ["# E14-A results", "", f"profile {PROFILE}, rerun {RERUN}", "", "## Decisions", ""]
    for _k, _v in (E14_RESULTS.get("decisions") or {}).items():
        _lines.append(f"- {_k}: adopt={_v.get('adopt', _v.get('ok'))} | reason: {_v.get('reason', _v.get('verdict'))} | "
                      f"{json.dumps(_v, default=str)[:400]}")
    _lines += ["", "## Duck run", "", f"- A1 reserve: {E14_RESULTS.get('a1_reserve')}",
               f"- clean start: {E14_RESULTS.get('duck_clean_start')}", f"- window: {E14_RESULTS.get('duck_window')}",
               f"- arm: {E14_RESULTS.get('duck_arm')}", "", "## Phases", ""]
    for _k, _v in (E14_RESULTS.get("phases") or {}).items():
        _lines.append(f"- {_k}: {json.dumps(_v, default=str)[:600]}")
    (E14_OUT / "e14_report.md").write_text("\n".join(_lines), encoding="utf-8")
    _pack = WORKING / "Q27X_E14A_BUNDLE.zip"
    with zipfile.ZipFile(_pack, "w", zipfile.ZIP_DEFLATED) as z:
        for fp in sorted(E14_OUT.rglob("*")):
            if fp.is_file() and fp.stat().st_size < 300e6:
                z.write(fp, arcname="e14/" + fp.relative_to(E14_OUT).as_posix())
        for fp in sorted(OUT.glob("sglang_server_*.log")):
            z.write(fp, arcname="logs/" + fp.name)
        for fp in (WORKING / "e14_state.json", WORKING / "e14_heartbeat.log", OUT / "e12_run_report.json"):
            if fp.exists():
                z.write(fp, arcname=fp.name)
    print("E14 report:", E14_OUT / "e14_report.md", "| bundle", _pack, round(_pack.stat().st_size / 1e6, 1), "MB", flush=True)
    print("E14 decisions:", json.dumps(E14_RESULTS.get("decisions"), default=str)[:2000], flush=True)
except BaseException:
    print("E14 report error:", traceback.format_exc()[-2000:], flush=True)
try:
    e11_kill_server(wait=False)
except Exception:
    pass
